# YOLO12m Competition Train V4

**한 번의 100 epoch 학습 trajectory**에서 20·40·60·100 checkpoint를 저장하고, 같은 Validation·같은 `mAP@[0.75:0.95]`·같은 confidence/Top-K grid로 비교한다.

`best.pt`(Ultralytics 기본 fitness)는 최종 선택 기준으로 사용하지 않는다.

## 실험 원칙

- 20/40/60/100을 각각 새로 학습하지 않는다.
- 총 100 epoch의 cosine LR trajectory를 딱 한 번 사용한다.
- 완료 epoch 20·40·60·100의 가중치를 별도 파일로 보존한다.
- checkpoint 선택과 confidence/Top-K 선택은 **Validation mAP@[0.75:0.95]**만 사용한다.
- 기존 mAP50-95/F1은 진단용이다.
- 최종 설정 확정 후에는 옵션으로 232장 전체를 같은 100-epoch LR horizon에서 재학습하고, 선택 epoch의 full-data checkpoint를 제출용으로 저장한다.

# 0. 기본세팅

## 0-1. 재현 가능한 패키지 설치

Colab의 CUDA와 연결된 `torch`, `torchvision`은 기본 버전을 유지한다. 모델 동작과 평가 결과에 직접 영향을 주는 패키지만 고정하고, 실제 버전은 뒤에서 실험 기록에 같이 저장한다.

In [ ]:
# 같은 노트북을 다시 실행해도 학습·평가 동작이 바뀌지 않도록 핵심 패키지 버전을 고정한다.
import subprocess
import sys

# YOLO12m 학습과 객체 탐지 평가를 지원하는 검증 버전이다.
PINNED_PACKAGES = [
    "ultralytics==8.4.116",
    "torchmetrics==1.9.0",
    "pycocotools==2.0.11",
    "albumentations==2.0.8",
    "tqdm==4.67.1",
]

# 현재 Colab Python에 필요한 패키지를 설치한다.
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *PINNED_PACKAGES])
print("Package installation completed.")

Package installation completed.


## 0-2. 라이브러리·Seed·환경 정보

In [ ]:
# 표준 라이브러리는 파일, 해시, 설정, 시간과 안전한 복사에 사용한다.
import gc
import hashlib
import importlib.metadata as importlib_metadata
import json
import math
import os
import platform
import random
import re
import shutil
import time
from collections import Counter, defaultdict
from datetime import datetime
from pathlib import Path

# 데이터 처리, 증강, 통계와 시각화에 필요한 라이브러리를 불러온다.
import albumentations as A
import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml
from PIL import Image, ImageDraw
from scipy.optimize import linear_sum_assignment
from tqdm.auto import tqdm

# PyTorch와 Torchvision의 객체 탐지 구성요소를 불러온다.
import torch
import torchvision
from torchvision.ops import box_iou

# 세 모델에 같은 COCO 방식 mAP를 적용하고 YOLO 모델을 실행한다.
from torchmetrics.detection.mean_ap import MeanAveragePrecision
import ultralytics
from ultralytics import YOLO

# Colab Drive와 표 출력을 사용한다.
from google.colab import drive
from IPython.display import display

# 전처리 산출물과 체크포인트가 저장된 Google Drive를 연결한다.
drive.mount("/content/drive")

# Python, NumPy, PyTorch의 난수를 하나의 Seed로 고정한다.
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# GPU가 있을 때 모든 CUDA 장치에도 같은 Seed를 적용한다.
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)

# 재현성을 우선하고 지원되지 않는 결정론 연산은 경고만 표시한다.
torch.use_deterministic_algorithms(True, warn_only=True)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# 모든 모델이 같은 학습 장치를 사용한다.
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = bool(torch.cuda.is_available())

# 실행 환경을 체크포인트와 최종 보고서에 남긴다.
RUNTIME_VERSIONS = {
    "python": platform.python_version(),
    "torch": torch.__version__,
    "torchvision": torchvision.__version__,
    "ultralytics": ultralytics.__version__,
    "torchmetrics": importlib_metadata.version("torchmetrics"),
    "pycocotools": importlib_metadata.version("pycocotools"),
    "albumentations": importlib_metadata.version("albumentations"),
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "cuda": torch.version.cuda,
    "device": str(DEVICE),
}

print(f"Training device: {DEVICE}")
display(pd.Series(RUNTIME_VERSIONS, name="version").to_frame())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Training device: cuda


,version
python,3.12.13
torch,2.11.0+cu128
torchvision,0.26.0+cu128
ultralytics,8.4.116
torchmetrics,1.9.0
pycocotools,2.0.11
albumentations,2.0.8
numpy,2.0.2
pandas,2.2.2
cuda,12.8


## 0-3. 공통 실험 설정과 경로

전처리 노트북에서 만든 v6.0 최종 폴더를 그대로 연결한다. 입력 크기는 전처리 결과와 같은 960으로 유지한다. 메모리가 부족하면 후보 설정의 batch만 낮추고, 입력 크기는 세 모델 모두 동일하게 유지한다.

In [ ]:
# 실제 epoch 비교는 standard로 실행한다. smoke는 연결 확인용으로만 사용한다.
RUN_PROFILE = "standard"  # "standard" 또는 "smoke"
# TARGET_EPOCHS는 삭제했다. 이 V4는 "100 epoch 한 번만 학습" 설계라서 학습 epoch 수를 바꿀 수
# 있는 값이 아니다(과거에 20/40/60/100을 각각 별도 노트북으로 학습하던 구조의 잔재로, 값을
# 바꿔도 실제 학습에는 전혀 반영되지 않는 죽은 변수였다). 실제 epoch 수는 아래
# FIXED_COMPARISON_CONFIG["epochs"]=100 한 곳에서만 정의된다.
TARGET_MODEL = "YOLO12m"
CHECKPOINT_EPOCHS = (20, 40, 60, 100)
COMPETITION_METRIC = "mAP@[0.75:0.95]"
COMPETITION_IOU_THRESHOLDS = [0.75, 0.80, 0.85, 0.90, 0.95]
CONFIDENCE_CANDIDATES = [0.001, 0.01, 0.03, 0.05, 0.10, 0.20, 0.30, 0.50]
TOP_K_CANDIDATES = [4, 6, 10, None]
RUN_FULL_DATA_FINAL_TRAIN = False

PIPELINE_VERSION = "4.0"
NOTEBOOK_PATCH_VERSION = "4.0-competition"
PIPELINE_CODE_VERSION = "4.0-yolo12m-single-trajectory-competition"
EXPECTED_IMAGES = 232
EXPECTED_OBJECTS = 771
EXPECTED_NUM_CLASSES = 56
EXPECTED_GROUPS = 114
EXPECTED_TARGET_SIZE = 960

# 그룹 단위 분할 비율과 탐색 횟수다.
VAL_RATIO = 0.15
TEST_RATIO = 0.15
SPLIT_SEARCH_TRIALS = 1024

# 세 모델이 공통으로 사용하는 입력·평가 조건이다.
IMAGE_SIZE = 960
RAW_PREDICTION_CONFIDENCE = 0.001
COMMON_CONFIDENCE_THRESHOLD = 0.25
EVAL_IOU_THRESHOLD = 0.50
NMS_IOU_THRESHOLD = 0.70
# 모델 추론에서 후보로 유지할 최대 객체 수다.
MAX_DETECTIONS = 300
# TorchMetrics/PyCOCOEval의 표준 평가 상한을 별도로 사용한다.
COCO_EVAL_MAX_DETECTIONS = 100
BOOTSTRAP_ITERATIONS = 500
VISUALIZATION_IMAGE_COUNT = 4

# 진행률 막대의 출력 형식을 통일한다.
PROGRESS_BAR_FORMAT = (
    "{desc}: {percentage:3.0f}%|{bar}| {n_fmt}/{total_fmt} "
    "[{elapsed}<{remaining}, {rate_fmt}]"
)

# v6.0 전처리 최종 산출물 경로다.
SHARED_ROOT = Path("/content/drive/MyDrive/baby_kangaroo")
# 🔍 확인 필요 (경로): 실행 가이드 PDF는 공통 전처리 데이터 참조 루트를
#   `/content/drive/MyDrive/baby_kangaroo/baby_kangaroo_cache/`로 통일하라고 명시하는데,
#   확인 방법: Colab에서 두 경로를 각각 `!ls`로 열어 실제로 존재하는 쪽, preprocessing_report.json이
#   있는 쪽이 어디인지 확인한다. baby_kangaroo_cache/ 쪽에 최신 v6.0 산출물이 있다면 아래 경로를
#   그쪽으로 교체하고, 지금 경로가 맞다면(즉 가이드 문서가 더 최신 캐시 구조를 가리키는 것이라면)
#   그대로 두되 팀에 경로 기준을 한 번 맞춰달라고 확인한다.
# 이름은 "frcnn"이지만 실제로는 FRCNN을 학습하지 않는다. 세 모델(YOLO11s/YOLO11m/YOLO12m 등)이
# 공통으로 참조하는 기준 COCO 정답·dataset/group fingerprint의 출처 경로일 뿐이다.
CANONICAL_DATA_DIR = (
    SHARED_ROOT / "데이터전처리" / #연준님 coco 전처리 경로
)
YOLO_TRACK_DIRS = {
    "YOLO12m": (
        SHARED_ROOT / "데이터전처리" #연준님 yolo전처리 경로
    ),
}

# 기존 통합 파이프라인 결과는 고정 설정을 읽는 기준으로만 사용한다.
BASE_MODEL_ARTIFACT_ROOT = SHARED_ROOT / "파이프라인" / "model_pipeline_v4_0"
REFERENCE_MODEL_REPORT_DIR = BASE_MODEL_ARTIFACT_ROOT / "reports"
REFERENCE_MODEL_MANIFEST_DIR = BASE_MODEL_ARTIFACT_ROOT / "manifests"

# epoch 비교 결과는 기존 세 모델 결과와 섞이지 않도록 별도 하위 폴더에 저장한다.
EPOCH_COMPARISON_ROOT = SHARED_ROOT / "파이프라인" / "yolo12m_competition_train_v4_0"
ARTIFACT_ROOT = EPOCH_COMPARISON_ROOT
CHECKPOINT_DIR = ARTIFACT_ROOT / "checkpoints"
MANIFEST_DIR = ARTIFACT_ROOT / "manifests"
REPORT_DIR = ARTIFACT_ROOT / "reports"
FIGURE_DIR = ARTIFACT_ROOT / "figures"
BACKUP_DIR = ARTIFACT_ROOT / "incompatible_backups"
TIMING_DIR = ARTIFACT_ROOT / "timings"

# 이미지와 학습 run은 Drive 병목을 피하기 위해 Colab 로컬 SSD에 둔다.
LOCAL_WORK_ROOT = Path("/content/baby_kangaroo_yolo12m_competition_v4_0")
LOCAL_IMAGE_ROOT = LOCAL_WORK_ROOT / "shared_images"
LOCAL_YOLO_DATASET = LOCAL_WORK_ROOT / "yolo_common"
LOCAL_RUNS_ROOT = LOCAL_WORK_ROOT / "runs"

# 필요한 결과 폴더를 미리 만든다.
for directory in [
    CHECKPOINT_DIR,
    MANIFEST_DIR,
    REPORT_DIR,
    TIMING_DIR,
    FIGURE_DIR,
    BACKUP_DIR,
    LOCAL_WORK_ROOT,
    LOCAL_RUNS_ROOT,
]:
    directory.mkdir(parents=True, exist_ok=True)

# 특정 실험을 다시 학습해야 할 때만 실험 ID를 넣는다.
FORCE_RETRAIN_EXPERIMENTS = set()

#   YOLO12m은 아직 한 번도 학습한적이 없으므로, 지금 이 상태로 실행하면
#   셀 28의 train_yolo_experiment가 재사용 가능한 checkpoint를 못 찾고 바로 FileNotFoundError로 멈춘다.
#   REQUIRE_EXISTING_CHECKPOINT를 False로 바꿔 실제 학습을 진행하고, 학습이 끝난 뒤에는 다시
#   True로 되돌려 재실행 시 실수로 재학습되지 않게 한다.

# 평가 재실행 중 실수로 장시간 재학습하는 일을 막는다.
REQUIRE_EXISTING_CHECKPOINT = False

# 설정 오류를 실행 초기에 차단한다.
if RUN_PROFILE not in {"standard", "smoke"}:
    raise ValueError("RUN_PROFILE must be 'standard' or 'smoke'.")
if IMAGE_SIZE != EXPECTED_TARGET_SIZE:
    raise ValueError("IMAGE_SIZE must remain 960 for a fair v6.0 comparison.")
if RUN_PROFILE == "standard" and not torch.cuda.is_available():
    raise RuntimeError("The standard training profile requires a Colab GPU runtime.")

print(f"Pipeline version: {PIPELINE_VERSION}")
print(f"Target model: {TARGET_MODEL}")
print("Target epochs: 100 (fixed, single-trajectory design)")
print(f"Run profile: {RUN_PROFILE}")
print(f"Artifact root: {ARTIFACT_ROOT}")

Pipeline version: 4.0
Target model: YOLO11s
Target epochs: 100
Run profile: standard
Artifact root: /content/drive/MyDrive/baby_kangaroo/파이프라인/yolo11s_competition_train_v4_0


## 0-4. 파일 지문·manifest·안전한 백업 공용 함수

In [ ]:
def stable_json_hash(value):
    """딕셔너리 순서와 관계없이 같은 JSON 내용에 같은 SHA256 지문을 만든다."""
    payload = json.dumps(
        value,
        ensure_ascii=False,
        sort_keys=True,
        separators=(",", ":"),
        default=str,
    ).encode("utf-8")
    return hashlib.sha256(payload).hexdigest()


def sha256_file(path, chunk_size=1024 * 1024):
    """큰 이미지와 체크포인트를 메모리에 한꺼번에 올리지 않고 SHA256을 계산한다."""
    digest = hashlib.sha256()
    with Path(path).open("rb") as file:
        while True:
            chunk = file.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()


def write_json_atomic(path, value):
    """런타임 중단 때 불완전한 JSON이 남지 않도록 임시 파일을 완성한 뒤 교체한다."""
    path = Path(path)
    temporary_path = path.with_suffix(path.suffix + ".tmp")
    temporary_path.write_text(
        json.dumps(value, ensure_ascii=False, indent=2, default=str),
        encoding="utf-8",
    )
    temporary_path.replace(path)


def backup_file(path, reason):
    """비호환 파일은 삭제하지 않고 해시가 포함된 이름으로 보존한다."""
    path = Path(path)
    if not path.is_file():
        return None
    short_hash = sha256_file(path)[:10]
    safe_reason = re.sub(r"[^A-Za-z0-9_-]+", "_", reason).strip("_")
    destination = BACKUP_DIR / f"{path.stem}_{safe_reason}_{short_hash}{path.suffix}"
    if not destination.exists():
        shutil.copy2(path, destination)
    print(f"Backed up incompatible file: {destination.name}")
    return destination


def build_checkpoint_manifest(model_name, base_weights, dataset_fingerprint, split_fingerprint, config):
    """체크포인트를 만든 데이터·분할·환경·학습 설정을 하나의 신분증으로 만든다."""
    manifest = {
        "schema_version": 3,
        "pipeline_version": PIPELINE_VERSION,
        "pipeline_code_version": PIPELINE_CODE_VERSION,
        "model_name": model_name,
        "base_weights": base_weights,
        "dataset_fingerprint": dataset_fingerprint,
        "split_fingerprint": split_fingerprint,
        "runtime_versions": RUNTIME_VERSIONS,
        "training_config": config,
    }
    manifest["run_signature"] = stable_json_hash(manifest)[:16]
    return manifest


def save_checkpoint_manifest(checkpoint_path, manifest_path, expected_manifest):
    """학습 완료 상태와 체크포인트 해시를 manifest에 기록한다."""
    stored_manifest = dict(expected_manifest)
    stored_manifest["training_completed"] = True
    stored_manifest["checkpoint_sha256"] = sha256_file(checkpoint_path)
    write_json_atomic(manifest_path, stored_manifest)


def checkpoint_is_compatible(checkpoint_path, manifest_path, expected_manifest):
    """완료된 체크포인트가 현재 실험 계약과 정확히 같을 때만 재사용한다."""
    checkpoint_path = Path(checkpoint_path)
    manifest_path = Path(manifest_path)
    if not checkpoint_path.is_file() or not manifest_path.is_file():
        return False

    try:
        saved_manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    except (OSError, json.JSONDecodeError):
        return False

    saved_checkpoint_hash = saved_manifest.pop("checkpoint_sha256", None)
    training_completed = bool(saved_manifest.pop("training_completed", False))
    if not training_completed or saved_checkpoint_hash != sha256_file(checkpoint_path):
        return False
    return saved_manifest == expected_manifest



def find_checkpoint_by_training_contract(experiment_id, expected_manifest):
    """런타임 버전만 달라진 완료 체크포인트를 학습 계약과 SHA256으로 찾는다."""
    contract_keys = [
        "schema_version",
        "pipeline_version",
        "pipeline_code_version",
        "model_name",
        "base_weights",
        "dataset_fingerprint",
        "split_fingerprint",
        "training_config",
    ]
    candidates = sorted(
        MANIFEST_DIR.glob(f"{experiment_id}_*.json"),
        key=lambda path: path.stat().st_mtime,
        reverse=True,
    )
    for candidate_manifest_path in candidates:
        try:
            stored_manifest = json.loads(
                candidate_manifest_path.read_text(encoding="utf-8")
            )
        except (OSError, json.JSONDecodeError):
            continue
        if not stored_manifest.get("training_completed"):
            continue
        if any(
            stored_manifest.get(key) != expected_manifest.get(key)
            for key in contract_keys
        ):
            continue

        stored_signature = stored_manifest.get("run_signature")
        stored_checkpoint_hash = stored_manifest.get("checkpoint_sha256")
        if not stored_signature or not stored_checkpoint_hash:
            continue
        candidate_checkpoint_path = (
            CHECKPOINT_DIR
            / f"{experiment_id}_{stored_signature}_best.pt"
        )
        if not candidate_checkpoint_path.is_file():
            continue
        if sha256_file(candidate_checkpoint_path) != stored_checkpoint_hash:
            raise RuntimeError(
                "Stored checkpoint SHA256 mismatch: "
                f"{candidate_checkpoint_path}"
            )
        return (
            candidate_checkpoint_path,
            candidate_manifest_path,
            stored_manifest,
        )
    return None

def reset_owned_directory(directory, allowed_root):
    """이 파이프라인 전용 로컬 폴더만 초기화해 잘못된 경로 삭제를 막는다."""
    directory = Path(directory).resolve()
    allowed_root = Path(allowed_root).resolve()
    if directory == allowed_root or allowed_root not in directory.parents:
        raise RuntimeError(f"Deletion is not allowed outside the owned root: {directory}")
    if directory.exists():
        shutil.rmtree(directory)
    directory.mkdir(parents=True, exist_ok=True)


def json_ready(value):
    """NumPy·Pandas·Path 값을 JSON으로 저장 가능한 기본 자료형으로 바꾼다."""
    if isinstance(value, dict):
        return {str(key): json_ready(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_ready(item) for item in value]
    if isinstance(value, np.integer):
        return int(value)
    if isinstance(value, np.floating):
        return float(value)
    if isinstance(value, Path):
        return str(value)
    return value

# 1. 데이터 - 결과물 받는 자리

## 1-0. 전처리 계약과 그룹 분할 공용 함수

연준님이 기존 파이프라인에서 같은 촬영 조합이 Train과 Validation에 섞이지 않도록 그룹 단위로 분할하고, 단일 그룹에만 존재하는 클래스는 Train에 남기는 방식을 세심하게 잡아두었다. 연준님이 이 기준을 먼저 정리해둔 덕분에 비슷한 사진이 서로 다른 분할에 들어가는 데이터 누수를 막으면서 희소 클래스도 보호할 수 있었다.

여기서는 그 설계 의도를 그대로 유지하면서 Test를 추가한다. 세 전처리 폴더의 `preprocessing_report.json`과 `dataset_manifest.csv`를 먼저 확인하고, 그룹 ID도 파일명에서 다시 추측하지 않고 전처리 결과에 저장된 값을 사용한다. 나머지 그룹으로 여러 분할 후보를 만든 뒤 클래스 커버리지가 가장 나은 후보를 선택한다.


In [ ]:
def read_preprocessing_report(bundle_dir):
    """v6.0 전처리 완료 상태와 핵심 계약을 읽고 검증한다."""
    bundle_dir = Path(bundle_dir)
    report_candidates = [
        bundle_dir / "preprocessing_report.json",
        bundle_dir / "final_report.json",
    ]
    report_path = next((path for path in report_candidates if path.is_file()), None)
    if report_path is None:
        raise FileNotFoundError(f"Preprocessing report was not found: {bundle_dir}")

    report = json.loads(report_path.read_text(encoding="utf-8"))
    if report.get("release_status") != "READY_FOR_EXTERNAL_GROUP_SPLIT":
        raise RuntimeError(f"Preprocessing release is not ready: {report_path}")
    if report.get("image_preprocessing_applied") is not True:
        raise RuntimeError(f"Image preprocessing was not confirmed: {report_path}")
    if int(report.get("preprocessing_target_size", -1)) != EXPECTED_TARGET_SIZE:
        raise RuntimeError(f"Unexpected preprocessing size: {report_path}")
    return report_path, report


def read_dataset_manifest(bundle_dir):
    """파일명·그룹·객체 수·전처리 상태가 담긴 데이터 manifest를 검증한다."""
    manifest_path = Path(bundle_dir) / "dataset_manifest.csv"
    if not manifest_path.is_file():
        raise FileNotFoundError(f"Dataset manifest was not found: {manifest_path}")

    manifest_df = pd.read_csv(manifest_path)
    required_columns = {"file_name", "group_id", "object_count"}
    missing_columns = required_columns - set(manifest_df.columns)
    if missing_columns:
        raise ValueError(f"Dataset manifest columns are missing: {sorted(missing_columns)}")
    if manifest_df["file_name"].duplicated().any():
        raise ValueError(f"Duplicate file names were found: {manifest_path}")
    if manifest_df["group_id"].isna().any():
        raise ValueError(f"Missing group IDs were found: {manifest_path}")
    return manifest_df.sort_values("file_name").reset_index(drop=True)


def _split_candidate_score(split_groups, group_files, group_class_counts, class_groups):
    """이미지 비율, 클래스 분포와 평가 클래스 누락을 합친 분할 점수를 계산한다."""
    target_ratios = {
        "train": 1.0 - VAL_RATIO - TEST_RATIO,
        "val": VAL_RATIO,
        "test": TEST_RATIO,
    }
    total_images = sum(len(files) for files in group_files.values())
    total_class_counts = Counter()
    for counts in group_class_counts.values():
        total_class_counts.update(counts)

    score = 0.0
    split_class_counts = {}
    for split_name in ["train", "val", "test"]:
        groups = split_groups[split_name]
        image_count = sum(len(group_files[group]) for group in groups)
        actual_ratio = image_count / max(total_images, 1)
        score += 25.0 * abs(actual_ratio - target_ratios[split_name])

        class_counts = Counter()
        for group in groups:
            class_counts.update(group_class_counts[group])
        split_class_counts[split_name] = class_counts

    for category_id, total_count in total_class_counts.items():
        for split_name in ["train", "val", "test"]:
            actual_ratio = split_class_counts[split_name][category_id] / max(total_count, 1)
            score += abs(actual_ratio - target_ratios[split_name])

        group_count = len(class_groups[category_id])
        if group_count >= 3:
            if split_class_counts["val"][category_id] == 0:
                score += 1000.0
            if split_class_counts["test"][category_id] == 0:
                score += 1000.0
        elif group_count == 2:
            if split_class_counts["val"][category_id] + split_class_counts["test"][category_id] == 0:
                score += 100.0
    return score


def build_group_train_val_test_split(coco, file_to_group):
    """전처리 manifest의 그룹 ID를 기준으로 누수 없는 세 분할을 만든다."""
    image_by_id = {image["id"]: image for image in coco["images"]}
    group_files = defaultdict(list)
    group_class_counts = defaultdict(Counter)
    class_groups = defaultdict(set)

    for image in coco["images"]:
        file_name = image["file_name"]
        if file_name not in file_to_group:
            raise KeyError(f"Group ID is missing for {file_name}")
        group_files[file_to_group[file_name]].append(file_name)

    for annotation in coco["annotations"]:
        image = image_by_id[annotation["image_id"]]
        group_id = file_to_group[image["file_name"]]
        category_id = int(annotation["category_id"])
        group_class_counts[group_id][category_id] += 1
        class_groups[category_id].add(group_id)

    all_groups = sorted(group_files)
    forced_train_groups = set()
    for category_id, groups in class_groups.items():
        if len(groups) == 1:
            forced_train_groups.update(groups)

    free_groups = [group for group in all_groups if group not in forced_train_groups]
    val_group_count = max(1, round(len(all_groups) * VAL_RATIO))
    test_group_count = max(1, round(len(all_groups) * TEST_RATIO))
    if val_group_count + test_group_count >= len(free_groups):
        raise ValueError("Not enough free groups for Validation and Test.")

    best_score = float("inf")
    best_split_groups = None
    for trial in tqdm(
        range(SPLIT_SEARCH_TRIALS),
        desc="[1-0] Searching group splits",
        unit="candidate",
        bar_format=PROGRESS_BAR_FORMAT,
    ):
        shuffled = list(free_groups)
        random.Random(SEED + trial).shuffle(shuffled)
        candidate = {
            "test": set(shuffled[:test_group_count]),
            "val": set(shuffled[test_group_count:test_group_count + val_group_count]),
            "train": set(shuffled[test_group_count + val_group_count:]) | forced_train_groups,
        }
        score = _split_candidate_score(
            candidate,
            group_files,
            group_class_counts,
            class_groups,
        )
        if score < best_score:
            best_score = score
            best_split_groups = candidate

    split_files = {}
    for split_name, groups in best_split_groups.items():
        split_files[split_name] = sorted(
            file_name
            for group in sorted(groups)
            for file_name in group_files[group]
        )

    for left, right in [("train", "val"), ("train", "test"), ("val", "test")]:
        if set(split_files[left]) & set(split_files[right]):
            raise RuntimeError(f"File leakage was found between {left} and {right}.")
        left_groups = {file_to_group[name] for name in split_files[left]}
        right_groups = {file_to_group[name] for name in split_files[right]}
        if left_groups & right_groups:
            raise RuntimeError(f"Group leakage was found between {left} and {right}.")

    assigned_files = set().union(*(set(files) for files in split_files.values()))
    expected_files = {image["file_name"] for image in coco["images"]}
    if assigned_files != expected_files:
        raise RuntimeError("The split contains missing or unexpected images.")

    return split_files, class_groups, best_score, best_split_groups

## 1-1. 공통 기준(Canonical) COCO 데이터 전체 검증

In [ ]:
def load_and_validate_coco_bundle(bundle_dir):
    """공통 기준 COCO JSON, 이미지, BBox와 v6.0 전처리 계약을 전부 검증한다."""
    bundle_dir = Path(bundle_dir)
    annotation_path = bundle_dir / "labels" / "annotations.json"
    image_dir = bundle_dir / "images"
    report_path, report = read_preprocessing_report(bundle_dir)
    manifest_df = read_dataset_manifest(bundle_dir)

    if not annotation_path.is_file():
        raise FileNotFoundError(f"COCO annotation was not found: {annotation_path}")
    if not image_dir.is_dir():
        raise FileNotFoundError(f"Canonical COCO image directory was not found: {image_dir}")

    with annotation_path.open(encoding="utf-8") as file:
        coco = json.load(file)

    required_keys = {"images", "annotations", "categories"}
    missing_keys = required_keys - set(coco)
    if missing_keys:
        raise KeyError(f"COCO keys are missing: {sorted(missing_keys)}")
    if len(coco["images"]) != EXPECTED_IMAGES:
        raise ValueError(f"Expected {EXPECTED_IMAGES} images, found {len(coco['images'])}.")
    if len(coco["annotations"]) != EXPECTED_OBJECTS:
        raise ValueError(f"Expected {EXPECTED_OBJECTS} objects, found {len(coco['annotations'])}.")
    if len(coco["categories"]) != EXPECTED_NUM_CLASSES:
        raise ValueError(f"Expected {EXPECTED_NUM_CLASSES} classes, found {len(coco['categories'])}.")
    if manifest_df["group_id"].nunique() != EXPECTED_GROUPS:
        raise ValueError(f"Expected {EXPECTED_GROUPS} groups, found {manifest_df['group_id'].nunique()}.")

    categories = coco["categories"]
    category_ids = [int(category["id"]) for category in categories]
    original_ids = [int(category["original_category_id"]) for category in categories]
    if len(category_ids) != len(set(category_ids)):
        raise ValueError("Duplicate COCO category IDs were found.")
    if len(original_ids) != len(set(original_ids)):
        raise ValueError("Duplicate original category IDs were found.")

    images = coco["images"]
    image_ids = [image["id"] for image in images]
    file_names = [image["file_name"] for image in images]
    if len(image_ids) != len(set(image_ids)) or len(file_names) != len(set(file_names)):
        raise ValueError("Duplicate COCO image IDs or file names were found.")
    if set(file_names) != set(manifest_df["file_name"]):
        raise ValueError("COCO images and dataset_manifest.csv do not have the same file names.")

    image_by_id = {image["id"]: image for image in images}
    category_id_set = set(category_ids)
    image_hashes = {}
    for image in tqdm(
        images,
        desc="[1-1] Validating COCO images",
        unit="image",
        bar_format=PROGRESS_BAR_FORMAT,
    ):
        image_path = image_dir / image["file_name"]
        if not image_path.is_file():
            raise FileNotFoundError(f"COCO image was not found: {image_path}")
        with Image.open(image_path) as pil_image:
            actual_size = pil_image.size
        expected_size = (int(image["width"]), int(image["height"]))
        if actual_size != expected_size or actual_size != (EXPECTED_TARGET_SIZE, EXPECTED_TARGET_SIZE):
            raise ValueError(f"Unexpected image size for {image['file_name']}: {actual_size}")
        image_hashes[image["file_name"]] = sha256_file(image_path)

    for index, annotation in enumerate(coco["annotations"]):
        if annotation["image_id"] not in image_by_id:
            raise ValueError(f"Unknown image_id in annotation {index}.")
        if int(annotation["category_id"]) not in category_id_set:
            raise ValueError(f"Unknown category_id in annotation {index}.")
        bbox = annotation.get("bbox")
        if not isinstance(bbox, list) or len(bbox) != 4:
            raise ValueError(f"Invalid BBox format in annotation {index}.")
        x, y, width, height = map(float, bbox)
        if not all(math.isfinite(value) for value in [x, y, width, height]):
            raise ValueError(f"Non-finite BBox value in annotation {index}.")
        if width <= 0 or height <= 0:
            raise ValueError(f"Non-positive BBox area in annotation {index}.")
        image = image_by_id[annotation["image_id"]]
        if x < -1e-6 or y < -1e-6 or x + width > image["width"] + 1e-6 or y + height > image["height"] + 1e-6:
            raise ValueError(f"Out-of-bounds BBox in annotation {index}.")

    dataset_fingerprint = stable_json_hash({
        "annotation_sha256": sha256_file(annotation_path),
        "image_hashes": image_hashes,
        "label_contract_sha256": report.get("label_contract_sha256"),
        "group_contract_sha256": report.get("group_contract_sha256"),
    })

    print(
        f"Canonical bundle verified: {len(images)} images, {len(coco['annotations'])} objects, "
        f"{len(categories)} classes, fingerprint {dataset_fingerprint[:12]}."
    )
    return {
        "coco": coco,
        "bundle_dir": bundle_dir,
        "annotation_path": annotation_path,
        "image_dir": image_dir,
        "manifest_df": manifest_df,
        "report_path": report_path,
        "report": report,
        "image_hashes": image_hashes,
        "dataset_fingerprint": dataset_fingerprint,
    }


# COCO 데이터를 세 모델의 기준 라벨로 먼저 검증한다.
canonical_data = load_and_validate_coco_bundle(CANONICAL_DATA_DIR)

[1-1] Validating COCO images:   0%|          | 0/232 [00:00<?, ?image/s]

FRCNN bundle verified: 232 images, 771 objects, 56 classes, fingerprint 36d6138cbb3b.


## 1-2. YOLO12m 데이터 전체 검증


In [ ]:
def read_yolo_mapping(mapping_path):
    """YOLO 0~55 ID와 실제 original_category_id의 1:1 매핑을 검증한다."""
    mapping_df = pd.read_csv(mapping_path)
    required_columns = {"yolo_class_id", "original_category_id", "normalized_class_name"}
    missing_columns = required_columns - set(mapping_df.columns)
    if missing_columns:
        raise ValueError(f"YOLO mapping columns are missing: {sorted(missing_columns)}")

    mapping_df = mapping_df.copy()
    mapping_df["yolo_class_id"] = pd.to_numeric(mapping_df["yolo_class_id"], errors="raise").astype(int)
    mapping_df["original_category_id"] = pd.to_numeric(
        mapping_df["original_category_id"], errors="raise"
    ).astype(int)
    mapping_df = mapping_df.sort_values("yolo_class_id").reset_index(drop=True)

    if mapping_df["yolo_class_id"].tolist() != list(range(EXPECTED_NUM_CLASSES)):
        raise ValueError(f"YOLO IDs must be continuous from 0 to {EXPECTED_NUM_CLASSES - 1}.")
    if mapping_df["original_category_id"].duplicated().any():
        raise ValueError("Duplicate original category IDs were found in the YOLO mapping.")
    if mapping_df["normalized_class_name"].fillna("").astype(str).str.strip().eq("").any():
        raise ValueError("Empty class names were found in the YOLO mapping.")

    idx_to_original = dict(zip(
        mapping_df["yolo_class_id"].astype(int),
        mapping_df["original_category_id"].astype(int),
    ))
    return mapping_df, idx_to_original


def validate_yolo_label_file(label_path):
    """YOLO TXT의 클래스와 정규화 좌표를 한 줄씩 검사하고 객체 목록을 반환한다."""
    records = []
    for line_number, raw_line in enumerate(Path(label_path).read_text(encoding="utf-8").splitlines(), start=1):
        if not raw_line.strip():
            continue
        fields = raw_line.split()
        if len(fields) != 5:
            raise ValueError(f"Invalid YOLO label format: {label_path}:{line_number}")
        class_value, center_x, center_y, width, height = map(float, fields)
        if not class_value.is_integer():
            raise ValueError(f"Non-integer YOLO class ID: {label_path}:{line_number}")
        class_id = int(class_value)
        if not 0 <= class_id < EXPECTED_NUM_CLASSES:
            raise ValueError(f"Out-of-range YOLO class ID: {label_path}:{line_number}")
        if not all(math.isfinite(value) for value in [center_x, center_y, width, height]):
            raise ValueError(f"Non-finite YOLO coordinate: {label_path}:{line_number}")
        if not (0 <= center_x <= 1 and 0 <= center_y <= 1 and 0 < width <= 1 and 0 < height <= 1):
            raise ValueError(f"Out-of-range YOLO coordinate: {label_path}:{line_number}")
        if center_x - width / 2 < -1e-5 or center_x + width / 2 > 1 + 1e-5:
            raise ValueError(f"YOLO x coordinates exceed the image: {label_path}:{line_number}")
        if center_y - height / 2 < -1e-5 or center_y + height / 2 > 1 + 1e-5:
            raise ValueError(f"YOLO y coordinates exceed the image: {label_path}:{line_number}")
        records.append((class_id, center_x, center_y, width, height))
    return records


def load_and_validate_yolo_bundle(track_name, bundle_dir, canonical_file_names):
    """YOLO 이미지·라벨·매핑·전처리 계약을 전체 검사하고 COCO 파일 목록과 맞춘다."""
    bundle_dir = Path(bundle_dir)
    image_dir = bundle_dir / "images"
    label_dir = bundle_dir / "labels"
    mapping_path = bundle_dir / "class_mapping.csv"
    report_path, report = read_preprocessing_report(bundle_dir)
    manifest_df = read_dataset_manifest(bundle_dir)

    for required_path in [image_dir, label_dir, mapping_path]:
        if not required_path.exists():
            raise FileNotFoundError(f"Required {track_name} output was not found: {required_path}")

    mapping_df, idx_to_original = read_yolo_mapping(mapping_path)
    canonical_original_ids = {
        int(category["original_category_id"]) for category in canonical_data["coco"]["categories"]
    }
    if set(mapping_df["original_category_id"].astype(int)) != canonical_original_ids:
        raise ValueError(f"{track_name} and COCO have different original category IDs.")

    image_files = sorted(path.name for path in image_dir.iterdir() if path.is_file())
    if set(image_files) != set(canonical_file_names):
        raise ValueError(f"{track_name} and COCO have different image file names.")
    if set(manifest_df["file_name"]) != set(canonical_file_names):
        raise ValueError(f"{track_name} manifest and COCO have different file names.")

    total_objects = 0
    image_hashes = {}
    label_records = {}
    for file_name in tqdm(
        canonical_file_names,
        desc=f"[1-2] Validating {track_name}",
        unit="image",
        bar_format=PROGRESS_BAR_FORMAT,
    ):
        image_path = image_dir / file_name
        label_path = label_dir / f"{Path(file_name).stem}.txt"
        if not label_path.is_file():
            raise FileNotFoundError(f"YOLO label was not found: {label_path}")
        with Image.open(image_path) as image:
            if image.size != (EXPECTED_TARGET_SIZE, EXPECTED_TARGET_SIZE):
                raise ValueError(f"Unexpected {track_name} image size: {file_name} -> {image.size}")
        records = validate_yolo_label_file(label_path)
        label_records[file_name] = records
        total_objects += len(records)
        image_hashes[file_name] = sha256_file(image_path)

    if total_objects != EXPECTED_OBJECTS:
        raise ValueError(f"Expected {EXPECTED_OBJECTS} {track_name} objects, found {total_objects}.")

    print(
        f"{track_name} bundle verified: {len(image_files)} images, {total_objects} objects, "
        f"{len(mapping_df)} classes."
    )
    return {
        "track_name": track_name,
        "bundle_dir": bundle_dir,
        "image_dir": image_dir,
        "label_dir": label_dir,
        "mapping_path": mapping_path,
        "mapping_df": mapping_df,
        "idx_to_original": idx_to_original,
        "manifest_df": manifest_df,
        "report_path": report_path,
        "report": report,
        "image_hashes": image_hashes,
        "label_records": label_records,
    }


# COCO 기준 파일 목록을 YOLO12m 전처리 결과에 적용한다.
canonical_file_names = [image["file_name"] for image in canonical_data["coco"]["images"]]
yolo_data = {
    track_name: load_and_validate_yolo_bundle(track_name, bundle_dir, canonical_file_names)
    for track_name, bundle_dir in YOLO_TRACK_DIRS.items()
}

[1-2] Validating YOLO11s:   0%|          | 0/232 [00:00<?, ?image/s]

YOLO11s bundle verified: 232 images, 771 objects, 56 classes.


## 1-3. COCO·YOLO12m 라벨과 그룹 계약 비교


In [ ]:
def load_coco_original_annotations(coco):
    """COCO 내부 ID를 실제 original_category_id로 바꾼 이미지별 xyxy 박스를 만든다."""
    category_to_original = {
        int(category["id"]): int(category["original_category_id"])
        for category in coco["categories"]
    }
    image_by_id = {image["id"]: image for image in coco["images"]}
    annotations_by_file = defaultdict(list)
    for annotation in coco["annotations"]:
        image = image_by_id[annotation["image_id"]]
        x, y, width, height = map(float, annotation["bbox"])
        annotations_by_file[image["file_name"]].append((
            category_to_original[int(annotation["category_id"])],
            [x, y, x + width, y + height],
        ))
    return annotations_by_file


def load_yolo_original_annotations(track_data):
    """YOLO 정규화 라벨을 original_category_id와 960×960 픽셀 xyxy 박스로 변환한다."""
    annotations_by_file = defaultdict(list)
    for file_name in canonical_file_names:
        for class_id, center_x, center_y, width, height in track_data["label_records"][file_name]:
            center_x *= EXPECTED_TARGET_SIZE
            center_y *= EXPECTED_TARGET_SIZE
            width *= EXPECTED_TARGET_SIZE
            height *= EXPECTED_TARGET_SIZE
            annotations_by_file[file_name].append((
                int(track_data["idx_to_original"][class_id]),
                [
                    center_x - width / 2,
                    center_y - height / 2,
                    center_x + width / 2,
                    center_y + height / 2,
                ],
            ))
    return annotations_by_file


def compare_annotation_sets_one_to_one(coco_annotations, yolo_annotations):
    """같은 클래스의 박스를 Hungarian matching으로 1:1 비교한다."""
    hard_mismatches = []
    geometry_warnings = []
    for file_name in tqdm(
        canonical_file_names,
        desc="[1-3] Comparing COCO and YOLO labels",
        unit="image",
        leave=False,
        bar_format=PROGRESS_BAR_FORMAT,
    ):
        coco_items = coco_annotations.get(file_name, [])
        yolo_items = yolo_annotations.get(file_name, [])
        if len(coco_items) != len(yolo_items):
            hard_mismatches.append({"file_name": file_name, "reason": "object_count"})
            continue
        if not coco_items:
            continue

        coco_labels = np.asarray([label for label, _ in coco_items], dtype=np.int64)
        yolo_labels = np.asarray([label for label, _ in yolo_items], dtype=np.int64)
        if Counter(coco_labels.tolist()) != Counter(yolo_labels.tolist()):
            hard_mismatches.append({"file_name": file_name, "reason": "class_composition"})
            continue

        coco_boxes = torch.tensor([box for _, box in coco_items], dtype=torch.float32)
        yolo_boxes = torch.tensor([box for _, box in yolo_items], dtype=torch.float32)
        iou_matrix = box_iou(coco_boxes, yolo_boxes).numpy()
        cost_matrix = 1.0 - iou_matrix
        cost_matrix[coco_labels[:, None] != yolo_labels[None, :]] = 1_000_000.0
        row_indices, column_indices = linear_sum_assignment(cost_matrix)

        for row_index, column_index in zip(row_indices, column_indices):
            pair_iou = float(iou_matrix[row_index, column_index])
            detail = {
                "file_name": file_name,
                "original_category_id": int(coco_labels[row_index]),
                "iou": pair_iou,
            }
            if coco_labels[row_index] != yolo_labels[column_index] or pair_iou < 0.50:
                hard_mismatches.append({**detail, "reason": "hard_geometry_or_class"})
            elif pair_iou < 0.90:
                geometry_warnings.append({**detail, "reason": "minor_geometry_difference"})
    return hard_mismatches, geometry_warnings


# COCO 정답 계약과 YOLO12m 전처리 계약이 같은지 확인한다.
contract_fields = ["label_contract_sha256", "group_contract_sha256"]
for field in contract_fields:
    contract_values = {
        canonical_data["report"].get(field),
        yolo_data["YOLO12m"]["report"].get(field),
    }
    if None in contract_values or len(contract_values) != 1:
        raise ValueError(f"COCO and YOLO12m preprocessing bundles have different {field} values.")

# 두 manifest의 파일별 group_id가 완전히 같은지 확인한다.
canonical_group_map = dict(zip(
    canonical_data["manifest_df"]["file_name"],
    canonical_data["manifest_df"]["group_id"].astype(str),
))
for track_name, track_data in yolo_data.items():
    track_group_map = dict(zip(
        track_data["manifest_df"]["file_name"],
        track_data["manifest_df"]["group_id"].astype(str),
    ))
    if track_group_map != canonical_group_map:
        raise ValueError(f"{track_name} has a different file-to-group mapping.")

# 이미지 바이트와 라벨 내용을 전수 비교한다.
coco_original_annotations = load_coco_original_annotations(canonical_data["coco"])
all_label_geometry_warnings = []
for track_name, track_data in yolo_data.items():
    image_mismatches = [
        file_name
        for file_name in canonical_file_names
        if canonical_data["image_hashes"][file_name] != track_data["image_hashes"][file_name]
    ]
    if image_mismatches:
        raise ValueError(f"{track_name} has different image bytes: {image_mismatches[:5]}")

    yolo_original_annotations = load_yolo_original_annotations(track_data)
    hard_mismatches, geometry_warnings = compare_annotation_sets_one_to_one(
        coco_original_annotations,
        yolo_original_annotations,
    )
    if hard_mismatches:
        raise ValueError(f"{track_name} has label mismatches: {hard_mismatches[:5]}")
    all_label_geometry_warnings.extend(
        {"track_name": track_name, **warning} for warning in geometry_warnings
    )
    print(f"{track_name} image and label equivalence passed for all {EXPECTED_IMAGES} images.")

# 경미한 소수점 차이가 있으면 숨기지 않고 보고서에 남긴다.
if all_label_geometry_warnings:
    warning_path = REPORT_DIR / f"label_geometry_warnings_{canonical_data['dataset_fingerprint'][:12]}.csv"
    pd.DataFrame(all_label_geometry_warnings).to_csv(warning_path, index=False, encoding="utf-8-sig")
    print(f"Minor label geometry warnings were saved: {warning_path}")

[1-3] Comparing COCO and YOLO labels:   0%|          | 0/232 [00:00<?, ?image/s]

YOLO11s image and label equivalence passed for all 232 images.


## 1-4. Train / Validation / Test 확정과 클래스 커버리지 보고

In [ ]:
# 검증된 COCO 라벨과 전처리 manifest의 group_id로 공통 분할을 만든다.
split_files, class_groups, split_score, split_groups = build_group_train_val_test_split(
    canonical_data["coco"],
    canonical_group_map,
)

# 분할 목록과 데이터 지문을 묶어 이후 모든 체크포인트가 같은 분할인지 확인한다.
split_manifest = {
    "schema_version": 3,
    "dataset_fingerprint": canonical_data["dataset_fingerprint"],
    "group_contract_sha256": canonical_data["report"]["group_contract_sha256"],
    "seed": SEED,
    "val_ratio": VAL_RATIO,
    "test_ratio": TEST_RATIO,
    "search_trials": SPLIT_SEARCH_TRIALS,
    "split_score": split_score,
    "train_files": split_files["train"],
    "val_files": split_files["val"],
    "test_files": split_files["test"],
}
split_fingerprint = stable_json_hash(split_manifest)
split_manifest_path = MANIFEST_DIR / f"split_{split_fingerprint[:16]}.json"
write_json_atomic(split_manifest_path, split_manifest)

# 사람이 바로 확인할 수 있는 파일 단위 CSV도 함께 저장한다.
split_rows = []
for split_name, file_names in split_files.items():
    for file_name in file_names:
        split_rows.append({
            "file_name": file_name,
            "group_id": canonical_group_map[file_name],
            "split": split_name,
        })
split_assignment_df = pd.DataFrame(split_rows).sort_values(["split", "group_id", "file_name"])
split_assignment_path = MANIFEST_DIR / f"split_assignments_{split_fingerprint[:16]}.csv"
split_assignment_df.to_csv(split_assignment_path, index=False, encoding="utf-8-sig")

# 모든 모델 결과 딕셔너리에 같은 파일 목록을 넣는다.
for track_data in [canonical_data, *yolo_data.values()]:
    for split_name in ["train", "val", "test"]:
        track_data[f"{split_name}_files"] = list(split_files[split_name])


def build_class_coverage_report(coco, split_files, class_groups):
    """56개 클래스의 분할별 객체 수와 평가 가능 여부를 표로 만든다."""
    image_to_split = {
        file_name: split_name
        for split_name, file_names in split_files.items()
        for file_name in file_names
    }
    image_by_id = {image["id"]: image for image in coco["images"]}
    counts = defaultdict(Counter)
    for annotation in coco["annotations"]:
        image = image_by_id[annotation["image_id"]]
        counts[int(annotation["category_id"])][image_to_split[image["file_name"]]] += 1

    rows = []
    for category in sorted(coco["categories"], key=lambda item: int(item["id"])):
        category_id = int(category["id"])
        rows.append({
            "coco_category_id": category_id,
            "original_category_id": int(category["original_category_id"]),
            "class_name": category["name"],
            "group_count": len(class_groups[category_id]),
            "train_objects": counts[category_id]["train"],
            "val_objects": counts[category_id]["val"],
            "test_objects": counts[category_id]["test"],
            "test_status": "observed" if counts[category_id]["test"] > 0 else "not_observed",
        })
    return pd.DataFrame(rows)


class_coverage_table = build_class_coverage_report(
    canonical_data["coco"],
    split_files,
    class_groups,
)
class_coverage_path = REPORT_DIR / f"class_split_coverage_{split_fingerprint[:16]}.csv"
class_coverage_table.to_csv(class_coverage_path, index=False, encoding="utf-8-sig")

test_observed_class_count = int(class_coverage_table["test_objects"].gt(0).sum())
print(
    f"Split completed: Train {len(split_files['train'])}, Validation {len(split_files['val'])}, "
    f"Test {len(split_files['test'])}."
)
print(f"Split fingerprint: {split_fingerprint[:16]}")
print(
    f"Test class coverage: {test_observed_class_count}/{EXPECTED_NUM_CLASSES} "
    f"({test_observed_class_count / EXPECTED_NUM_CLASSES:.1%})."
)
display(class_coverage_table)

[1-0] Searching group splits:   0%|          | 0/1024 [00:00<?, ?candidate/s]

Split completed: Train 160, Validation 38, Test 34.
Split fingerprint: 3b914e2658165daa
Test class coverage: 28/56 (50.0%).


,coco_category_id,original_category_id,class_name,group_count,train_objects,val_objects,test_objects,test_status
0,1,1900,보령부스파정 5mg,5,15,0,0,not_observed
1,2,2483,뮤테란캡슐 100mg,3,9,0,0,not_observed
2,3,3351,일양하이트린정 2mg,89,106,29,22,observed
3,4,3483,기넥신에프정(은행엽엑스)(수출용),15,24,9,12,observed
4,5,3544,무코스타정(레바미피드)(비매품),2,6,0,0,not_observed
5,6,3743,알드린정,1,3,0,0,not_observed
6,7,3832,뉴로메드정(옥시라세탐),10,15,3,2,observed
7,8,4543,에어탈정(아세클로페낙),2,6,0,0,not_observed
8,9,12081,리렉스펜정 300mg/PTP,1,3,0,0,not_observed
9,10,12247,아빌리파이정 10mg,1,3,0,0,not_observed


## 1-5. YOLO12m 로컬 학습 데이터 준비


In [ ]:
def build_canonical_yolo_mapping(coco):
    """원본 category ID를 공통 YOLO 0~55 ID와 이름에 연결한다."""
    categories = sorted(coco["categories"], key=lambda item: int(item["original_category_id"]))
    original_to_yolo = {
        int(category["original_category_id"]): index
        for index, category in enumerate(categories)
    }
    names = {index: category["name"] for index, category in enumerate(categories)}
    return original_to_yolo, names


def write_canonical_yolo_label(file_name, destination_path, original_to_yolo):
    """공통 COCO xyxy 박스를 YOLO 정규화 형식으로 변환해 저장한다."""
    lines = []
    for original_id, box_xyxy in coco_original_annotations.get(file_name, []):
        x1, y1, x2, y2 = map(float, box_xyxy)
        center_x = ((x1 + x2) / 2.0) / EXPECTED_TARGET_SIZE
        center_y = ((y1 + y2) / 2.0) / EXPECTED_TARGET_SIZE
        width = (x2 - x1) / EXPECTED_TARGET_SIZE
        height = (y2 - y1) / EXPECTED_TARGET_SIZE
        lines.append(
            f"{original_to_yolo[int(original_id)]} "
            f"{center_x:.10f} {center_y:.10f} {width:.10f} {height:.10f}"
        )
    text = "\n".join(lines) + ("\n" if lines else "")
    Path(destination_path).write_text(text, encoding="utf-8")


def stage_shared_training_dataset():
    """YOLO12m 학습에 사용할 로컬 이미지와 라벨 폴더를 만든다."""
    reset_owned_directory(LOCAL_IMAGE_ROOT, LOCAL_WORK_ROOT)
    reset_owned_directory(LOCAL_YOLO_DATASET, LOCAL_WORK_ROOT)

    # 검증된 YOLO12m 전처리 이미지를 로컬에 한 번만 복사한다.
    for file_name in tqdm(
        canonical_file_names,
        desc="[1-5] Copying shared images",
        unit="image",
        bar_format=PROGRESS_BAR_FORMAT,
    ):
        shutil.copy2(yolo_data["YOLO12m"]["image_dir"] / file_name, LOCAL_IMAGE_ROOT / file_name)

    original_to_yolo, names = build_canonical_yolo_mapping(canonical_data["coco"])
    for split_name in ["train", "val", "test"]:
        image_split_dir = LOCAL_YOLO_DATASET / "images" / split_name
        label_split_dir = LOCAL_YOLO_DATASET / "labels" / split_name
        image_split_dir.mkdir(parents=True, exist_ok=True)
        label_split_dir.mkdir(parents=True, exist_ok=True)

        for file_name in split_files[split_name]:
            source_image = LOCAL_IMAGE_ROOT / file_name
            destination_image = image_split_dir / file_name
            # 같은 로컬 파일시스템에서는 hard link로 디스크 중복을 피한다.
            try:
                os.link(source_image, destination_image)
            except OSError:
                shutil.copy2(source_image, destination_image)
            write_canonical_yolo_label(
                file_name,
                label_split_dir / f"{Path(file_name).stem}.txt",
                original_to_yolo,
            )

    data_yaml = {
        "path": str(LOCAL_YOLO_DATASET),
        "train": "images/train",
        "val": "images/val",
        "test": "images/test",
        "nc": EXPECTED_NUM_CLASSES,
        "names": names,
    }
    data_yaml_path = LOCAL_YOLO_DATASET / "data.yaml"
    with data_yaml_path.open("w", encoding="utf-8") as file:
        yaml.safe_dump(data_yaml, file, allow_unicode=True, sort_keys=False)

    # 복사·hard link 이후에도 모든 이미지 해시가 원본 전처리 결과와 같은지 확인한다.
    staged_hashes = {
        file_name: sha256_file(LOCAL_IMAGE_ROOT / file_name)
        for file_name in canonical_file_names
    }
    if staged_hashes != yolo_data["YOLO12m"]["image_hashes"]:
        raise RuntimeError("Staged image hashes do not match the preprocessing bundle.")
    return data_yaml_path, original_to_yolo, names


data_yaml_path, ORIGINAL_TO_YOLO, YOLO_NAMES = stage_shared_training_dataset()
YOLO_TO_ORIGINAL = {yolo_id: original_id for original_id, yolo_id in ORIGINAL_TO_YOLO.items()}

print(f"Shared local image directory: {LOCAL_IMAGE_ROOT}")
print(f"Shared YOLO dataset: {data_yaml_path}")

[1-5] Copying shared images:   0%|          | 0/232 [00:00<?, ?image/s]

Shared local image directory: /content/baby_kangaroo_yolo11s_competition_v4_0/shared_images
Shared YOLO dataset: /content/baby_kangaroo_yolo11s_competition_v4_0/yolo_common/data.yaml


# 2. YOLO12m 학습


## 2-1. 고정 설정과 실험 타이머


In [ ]:
# 🔍 확인 필요 (하이퍼파라미터 기준): 아래 함수는 YOLO11s용으로 Validation에서 선택된
#   batch_size/learning_rate/momentum/weight_decay/augmentation을 그대로 읽어와 YOLO12m 학습에도 쓴다.
#   가이드 PDF의 "세 노트북은 항목을 동일하게 유지한다" 원칙에 따른 의도된 설계이지만, YOLO12m은
#   구조가 달라(attention 기반) 같은 batch_size에서 OOM이 날 수 있고, 팀이 YOLO12m 전용으로 다시
#   튜닝한 설정이 있을 수도 있다.
#   확인 방법: REFERENCE_MODEL_REPORT_DIR 안의 report json을 열어 "selected_experiments"에
#   "YOLO12m" 키가 이미 있는지 확인한다. 있다면 아래 함수가 "YOLO11s" 대신 그 키를 읽도록 바꾸고,
#   없다면 이 YOLO11s 기준값을 그대로 쓸지 팀(특히 파이프라인 담당)에 한 번 확인한다.
def load_reference_yolo11_config():
    report_paths = sorted(
        REFERENCE_MODEL_REPORT_DIR.glob("model_development_report_*.json"), # 여기
        key=lambda path: path.stat().st_mtime,
        reverse=True,
    )
    for report_path in report_paths:
        try:
            report = json.loads(report_path.read_text(encoding="utf-8"))
        except (OSError, json.JSONDecodeError):
            continue
        if report.get("run_profile") != "standard":
            continue
        selected_id = report.get("selected_experiments", {}).get("YOLO11s")
        checkpoint_text = report.get("checkpoint_paths", {}).get("YOLO11s")
        if not selected_id or not checkpoint_text:
            continue
        checkpoint_path = Path(checkpoint_text)
        prefix = f"{selected_id}_"
        suffix = "_best.pt"
        if not checkpoint_path.name.startswith(prefix) or not checkpoint_path.name.endswith(suffix):
            continue
        signature = checkpoint_path.name[len(prefix):-len(suffix)]
        manifest_path = REFERENCE_MODEL_MANIFEST_DIR / f"{selected_id}_{signature}.json"
        if not checkpoint_path.is_file() or not manifest_path.is_file():
            continue
        manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
        if not manifest.get("training_completed"):
            continue
        if manifest.get("checkpoint_sha256") != sha256_file(checkpoint_path):
            raise RuntimeError(f"Reference checkpoint SHA256 mismatch: {checkpoint_path}")
        training_config = dict(manifest.get("training_config", {}))
        required = {
            "stage", "batch_size", "learning_rate", "momentum",
            "weight_decay", "augmentation",
        }
        missing = required - set(training_config)
        if missing:
            raise KeyError(f"Reference YOLO11s config keys missing: {sorted(missing)}")
        return report_path, selected_id, checkpoint_path, manifest_path, training_config
    raise FileNotFoundError(
        "A completed standard YOLO11s reference report was not found in model_pipeline_v4_0. "
        "(This still looks for the YOLO11s reference on purpose — see the comment above load_reference_yolo11_config.)"
    )


(
    REFERENCE_REPORT_PATH,
    REFERENCE_EXPERIMENT_ID,
    REFERENCE_CHECKPOINT_PATH,
    REFERENCE_MANIFEST_PATH,
    REFERENCE_TRAINING_CONFIG,
) = load_reference_yolo11_config()

# 세 모델(YOLO11s/11m/12m) 노트북은 아래 항목을 동일하게 유지한다. epoch는 100 하나로 고정이며,
# 노트북별로 다르게 두는 항목은 모델 이름/가중치뿐이다.
FIXED_COMPARISON_CONFIG = {
    "experiment_id": "yolo12m_competition_100e_single_trajectory",
    "stage": REFERENCE_TRAINING_CONFIG["stage"],
    "epochs": 100,
    "batch_size": int(REFERENCE_TRAINING_CONFIG["batch_size"]),
    "learning_rate": float(REFERENCE_TRAINING_CONFIG["learning_rate"]),
    "momentum": float(REFERENCE_TRAINING_CONFIG["momentum"]),
    "weight_decay": float(REFERENCE_TRAINING_CONFIG["weight_decay"]),
    "augmentation": str(REFERENCE_TRAINING_CONFIG["augmentation"]),
    # 모든 비교 실험이 계획된 epoch를 끝까지 실행하도록 조기 종료 여유를 둔다.
    "early_stopping_patience": 100,
    "reference_experiment_id": REFERENCE_EXPERIMENT_ID,
}
BASELINE_CONFIGS = {"YOLO12m": FIXED_COMPARISON_CONFIG}
TUNING_CONFIGS = {"YOLO12m": []}


def effective_config(config):
    """smoke 모드에서만 epoch를 1로 줄이고 standard 설정은 그대로 유지한다."""
    resolved = dict(config)
    if RUN_PROFILE == "smoke":
        resolved["epochs"] = 1
        resolved["early_stopping_patience"] = 1
    resolved["image_size"] = IMAGE_SIZE
    resolved["run_profile"] = RUN_PROFILE
    return resolved


TIMING_SCHEMA_VERSION = 1


def format_duration(seconds):
    """초 단위 시간을 사람이 바로 읽을 수 있는 시:분:초 문자열로 바꾼다."""
    seconds = max(0.0, float(seconds))
    hours, remainder = divmod(int(round(seconds)), 3600)
    minutes, seconds_part = divmod(remainder, 60)
    return f"{hours:02d}:{minutes:02d}:{seconds_part:02d}"


def run_timed_experiment(timer_model_name, raw_config, train_callable, *args, **kwargs):
    """학습 함수의 시작·종료·실패 시간을 기록하고 실험별 JSON을 즉시 저장한다."""
    resolved_config = effective_config(raw_config)
    experiment_id = resolved_config["experiment_id"]
    started_at = datetime.now()
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    started_perf = time.perf_counter()
    print(f"Timer started: {timer_model_name} / {experiment_id} / {started_at.isoformat()}")

    try:
        result = train_callable(*args, **kwargs)
    except Exception as error:
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        elapsed_seconds = time.perf_counter() - started_perf
        finished_at = datetime.now()
        failed_record = {
            "schema_version": TIMING_SCHEMA_VERSION,
            "model": timer_model_name,
            "experiment_id": experiment_id,
            "stage": resolved_config["stage"],
            "run_profile": RUN_PROFILE,
            "last_execution_status": "failed",
            "last_execution_started_at": started_at.isoformat(),
            "last_execution_finished_at": finished_at.isoformat(),
            "last_execution_elapsed_seconds": elapsed_seconds,
            "last_execution_elapsed_hms": format_duration(elapsed_seconds),
            "error_type": type(error).__name__,
            "error_message": str(error),
        }
        failed_path = TIMING_DIR / (
            f"{experiment_id}_failed_{finished_at.strftime('%Y%m%d_%H%M%S')}.json"
        )
        write_json_atomic(failed_path, failed_record)
        print(f"Timer saved after failure: {failed_path}")
        raise

    if torch.cuda.is_available():
        torch.cuda.synchronize()
    elapsed_seconds = time.perf_counter() - started_perf
    finished_at = datetime.now()
    signature = result["manifest"]["run_signature"]
    timing_path = TIMING_DIR / f"{experiment_id}_{signature}.json"
    checkpoint_reused = bool(result.get("checkpoint_reused", False))

    if checkpoint_reused and timing_path.is_file():
        try:
            timing_record = json.loads(timing_path.read_text(encoding="utf-8"))
        except (OSError, json.JSONDecodeError):
            timing_record = {}
    else:
        timing_record = {}

    # 실제 학습이 수행된 실행만 원본 학습시간으로 고정한다. 재사용 시간은 별도 필드에 남긴다.
    if not checkpoint_reused:
        timing_record.update({
            "schema_version": TIMING_SCHEMA_VERSION,
            "model": timer_model_name,
            "experiment_id": experiment_id,
            "stage": resolved_config["stage"],
            "run_profile": RUN_PROFILE,
            "run_signature": signature,
            "training_started_at": started_at.isoformat(),
            "training_finished_at": finished_at.isoformat(),
            "training_elapsed_seconds": elapsed_seconds,
            "training_elapsed_minutes": elapsed_seconds / 60.0,
            "training_elapsed_hours": elapsed_seconds / 3600.0,
            "training_elapsed_hms": format_duration(elapsed_seconds),
            "reuse_count": 0,
        })
    else:
        timing_record.setdefault("schema_version", TIMING_SCHEMA_VERSION)
        timing_record.setdefault("model", timer_model_name)
        timing_record.setdefault("experiment_id", experiment_id)
        timing_record.setdefault("stage", resolved_config["stage"])
        timing_record.setdefault("run_profile", RUN_PROFILE)
        timing_record.setdefault("run_signature", signature)
        timing_record.setdefault("training_elapsed_seconds", None)
        timing_record.setdefault("training_elapsed_minutes", None)
        timing_record.setdefault("training_elapsed_hours", None)
        timing_record.setdefault("training_elapsed_hms", None)
        timing_record["reuse_count"] = int(timing_record.get("reuse_count", 0)) + 1

    timing_record.update({
        "checkpoint_reused_on_last_execution": checkpoint_reused,
        "last_execution_status": "checkpoint_reused" if checkpoint_reused else "completed",
        "last_execution_started_at": started_at.isoformat(),
        "last_execution_finished_at": finished_at.isoformat(),
        "last_execution_elapsed_seconds": elapsed_seconds,
        "last_execution_elapsed_hms": format_duration(elapsed_seconds),
        "timing_path": str(timing_path),
    })
    write_json_atomic(timing_path, timing_record)
    result["timing"] = timing_record
    print(
        f"Timer completed: {timer_model_name} / {experiment_id} / "
        f"{format_duration(elapsed_seconds)} / "
        f"status={timing_record['last_execution_status']}"
    )
    print(f"Timer record: {timing_path}")
    return result



# 실행 전에 고정 설정을 표로 확인한다.
experiment_plan_table = pd.DataFrame([{"model": "YOLO12m", **effective_config(FIXED_COMPARISON_CONFIG)}])
display(experiment_plan_table[[
    "model", "experiment_id", "stage", "epochs", "batch_size",
    "learning_rate", "weight_decay", "augmentation", "reference_experiment_id",
]])
print(f"Reference report: {REFERENCE_REPORT_PATH}")
print(f"Fixed comparison config: batch={FIXED_COMPARISON_CONFIG['batch_size']}, "
      f"lr={FIXED_COMPARISON_CONFIG['learning_rate']}, epochs={FIXED_COMPARISON_CONFIG['epochs']}")


,model,experiment_id,stage,epochs,batch_size,learning_rate,weight_decay,augmentation,reference_experiment_id
0,YOLO11s,yolo11s_competition_100e_single_trajectory,baseline,100,2,0.005,0.0005,baseline,yolo11s_baseline


Reference report: /content/drive/MyDrive/baby_kangaroo/파이프라인/model_pipeline_v4_0/reports/model_development_report_20260811_010238_3b914e26.json
Fixed comparison config: batch=2, lr=0.005, epochs=100


## 2-2. YOLO12m 공통 학습 함수


In [ ]:
def analyze_yolo_convergence(results_csv_path):
    """results.csv에서 best epoch, 마지막 추세와 수렴 부족 가능성을 계산한다."""
    results_df = pd.read_csv(results_csv_path)
    results_df.columns = [str(column).strip() for column in results_df.columns]
    metric_candidates = ["metrics/mAP50-95(B)", "metrics/mAP50-95"]
    metric_column = next((column for column in metric_candidates if column in results_df.columns), None)
    if metric_column is None:
        raise ValueError("Validation mAP50-95 was not found in the YOLO results.csv file.")

    metric_values = pd.to_numeric(results_df[metric_column], errors="raise").astype(float)
    if metric_values.empty or not np.isfinite(metric_values).all() or (metric_values < 0).any():
        raise RuntimeError("YOLO validation mAP contains invalid values.")
    best_index = int(metric_values.idxmax())
    tail_size = min(5, len(metric_values))
    tail_values = metric_values.iloc[-tail_size:].to_numpy()
    tail_slope = float(np.polyfit(np.arange(tail_size), tail_values, 1)[0]) if tail_size >= 2 else 0.0

    loss_columns = [
        column for column in results_df.columns
        if column.startswith("train/") and column.endswith("_loss")
    ]
    train_loss = (
        results_df[loss_columns].apply(pd.to_numeric, errors="coerce").sum(axis=1)
        if loss_columns else pd.Series(dtype=float)
    )
    near_end = best_index >= max(0, math.floor(len(metric_values) * 0.90) - 1)
    needs_more_epochs = bool(near_end and tail_slope > 0.0005)
    return {
        "best_epoch": best_index + 1,
        "best_val_map50_95": float(metric_values.iloc[best_index]),
        "last_val_map50_95": float(metric_values.iloc[-1]),
        "last_five_slope": tail_slope,
        "needs_more_epochs": needs_more_epochs,
        "train_loss_history": train_loss.astype(float).tolist(),
        "val_map50_95_history": metric_values.tolist(),
    }


def validate_yolo_model_scale(model, expected_scale, model_name):
    """체크포인트 내부 YAML의 모델 scale이 기대한 small인지 확인한다."""
    model_yaml = getattr(getattr(model, "model", None), "yaml", {}) or {}
    detected_scale = model_yaml.get("scale")
    if detected_scale is not None and str(detected_scale) != expected_scale:
        raise ValueError(
            f"{model_name} scale mismatch: expected {expected_scale}, found {detected_scale}."
        )
    print(f"{model_name} scale check: {detected_scale or 'metadata unavailable'}")


def yolo_augmentation_arguments(mode):
    """공통 기준과 강도가 비슷한 YOLO Train 전용 증강 설정을 반환한다."""
    if mode == "baseline":
        return {
            "hsv_h": 0.0,
            "hsv_s": 0.0,
            "hsv_v": 0.0,
            "degrees": 0.0,
            "translate": 0.0,
            "scale": 0.0,
            "shear": 0.0,
            "perspective": 0.0,
            "flipud": 0.0,
            "fliplr": 0.50,
            "mosaic": 0.0,
            "mixup": 0.0,
            "copy_paste": 0.0,
            "erasing": 0.0,
        }
    if mode == "safe_detection":
        return {
            "hsv_h": 0.008,
            "hsv_s": 0.10,
            "hsv_v": 0.08,
            "degrees": 8.0,
            "translate": 0.04,
            "scale": 0.05,
            "shear": 2.0,
            "perspective": 0.0,
            "flipud": 0.0,
            "fliplr": 0.50,
            "mosaic": 0.0,
            "mixup": 0.0,
            "copy_paste": 0.0,
            "erasing": 0.0,
        }
    raise ValueError(f"Unknown YOLO augmentation mode: {mode}")


def train_yolo_experiment(model_name, model_weights, expected_scale, raw_config):
    """YOLO12m 한 실험을 학습하고 best checkpoint와 설정을 저장한다."""
    config = effective_config(raw_config)
    experiment_id = config["experiment_id"]
    manifest = build_checkpoint_manifest(
        model_name,
        model_weights,
        canonical_data["dataset_fingerprint"],
        split_fingerprint,
        config,
    )
    signature = manifest["run_signature"]
    checkpoint_path = CHECKPOINT_DIR / f"{experiment_id}_{signature}_best.pt"
    manifest_path = MANIFEST_DIR / f"{experiment_id}_{signature}.json"
    results_path = CHECKPOINT_DIR / f"{experiment_id}_{signature}_results.csv"
    args_path = CHECKPOINT_DIR / f"{experiment_id}_{signature}_args.yaml"

    if experiment_id not in FORCE_RETRAIN_EXPERIMENTS and checkpoint_is_compatible(
        checkpoint_path,
        manifest_path,
        manifest,
    ):
        trained_model = YOLO(str(checkpoint_path))
        validate_yolo_model_scale(trained_model, expected_scale, model_name)
        convergence = analyze_yolo_convergence(results_path) if results_path.is_file() else None
        print(f"Reused verified {model_name} checkpoint: {checkpoint_path.name}")
        return {
            "model": trained_model,
            "model_name": model_name,
            "checkpoint_path": checkpoint_path,
            "manifest_path": manifest_path,
            "manifest": manifest,
            "config": config,
            "checkpoint_reused": True,
            "convergence": convergence,
            "results_path": results_path,
        }


    # Colab 기본 패키지 버전만 달라진 경우에도 동일 학습 계약의 완료 모델을 재사용한다.
    reusable_checkpoint = find_checkpoint_by_training_contract(
        experiment_id,
        manifest,
    )
    if reusable_checkpoint is not None:
        checkpoint_path, manifest_path, stored_manifest = reusable_checkpoint
        signature = stored_manifest["run_signature"]
        results_path = CHECKPOINT_DIR / f"{experiment_id}_{signature}_results.csv"
        trained_model = YOLO(str(checkpoint_path))
        validate_yolo_model_scale(trained_model, expected_scale, model_name)
        convergence = (
            analyze_yolo_convergence(results_path)
            if results_path.is_file()
            else None
        )
        reusable_manifest = dict(stored_manifest)
        reusable_manifest.pop("training_completed", None)
        reusable_manifest.pop("checkpoint_sha256", None)
        print(
            f"Reused verified {model_name} checkpoint by training contract: "
            f"{checkpoint_path.name}"
        )
        return {
            "model": trained_model,
            "model_name": model_name,
            "checkpoint_path": checkpoint_path,
            "manifest_path": manifest_path,
            "manifest": reusable_manifest,
            "config": config,
            "checkpoint_reused": True,
            "convergence": convergence,
            "results_path": results_path,
        }

    # 평가 수정본은 기존 학습 결과만 사용하며 실수로 새 학습을 시작하지 않는다.
    if REQUIRE_EXISTING_CHECKPOINT:
        raise FileNotFoundError(
            "A completed checkpoint matching the dataset, split, model, and "
            f"training config was not found for {experiment_id}. "
            f"Checked checkpoint directory: {CHECKPOINT_DIR}. "
            "Set REQUIRE_EXISTING_CHECKPOINT=False only when retraining is intended."
        )

    if checkpoint_path.exists():
        backup_file(checkpoint_path, "incomplete_or_incompatible")

    # 🔍 확인 필요 (배치 크기): YOLO12m은 attention 기반 구조라 같은 IMAGE_SIZE=960·batch_size에서도
    #   YOLO11s보다 GPU 메모리를 더 많이 쓸 수 있다. 확인 방법: 아래 model.train() 첫 실행에서
    #   CUDA out of memory가 나면, IMAGE_SIZE(=960, 유지해야 하는 계약값)는 그대로 두고
    #   config["batch_size"](load_reference_yolo11_config에서 가져온 값)만 낮춰서 재시도한다.
    model = YOLO(model_weights)
    validate_yolo_model_scale(model, expected_scale, model_name)
    run_name = f"{experiment_id}_{signature}_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
    augmentation_arguments = yolo_augmentation_arguments(config["augmentation"])

    model.train(
        data=str(data_yaml_path),
        epochs=config["epochs"],
        batch=config["batch_size"],
        imgsz=IMAGE_SIZE,
        device=0 if torch.cuda.is_available() else "cpu",
        optimizer="SGD",
        lr0=config["learning_rate"],
        lrf=0.10,
        momentum=config["momentum"],
        weight_decay=config["weight_decay"],
        warmup_epochs=2.0 if config["stage"] == "tuning" else 1.0,
        cos_lr=True,
        patience=config["early_stopping_patience"],
        max_det=MAX_DETECTIONS,
        seed=SEED,
        deterministic=True,
        workers=2,
        cache=False,
        amp=USE_AMP,
        project=str(LOCAL_RUNS_ROOT),
        name=run_name,
        exist_ok=False,
        plots=True,
        save=True,
        save_period=1,
        verbose=True,
        **augmentation_arguments,
    )

    run_dir = Path(model.trainer.save_dir)
    best_source = run_dir / "weights" / "best.pt"
    last_source = run_dir / "weights" / "last.pt"
    results_source = run_dir / "results.csv"
    args_source = run_dir / "args.yaml"
    for required_path in [best_source, last_source, results_source, args_source]:
        if not required_path.is_file():
            raise FileNotFoundError(f"Required {model_name} training output was not found: {required_path}")

    # V4에서는 Ultralytics best.pt가 아니라 100 epoch trajectory의 마지막 모델을 임시 기준으로 보존한다.
    shutil.copy2(last_source, checkpoint_path)
    shutil.copy2(results_source, results_path)
    shutil.copy2(args_source, args_path)
    # Ultralytics save_period 파일명은 내부 epoch index를 사용할 수 있으므로 완료 epoch를 +1로 해석한다.
    candidate_checkpoint_paths = {}
    weights_dir = run_dir / "weights"
    for periodic_path in sorted(weights_dir.glob("epoch*.pt")):
        match = re.fullmatch(r"epoch(\d+)\.pt", periodic_path.name)
        if not match:
            continue
        completed_epoch = int(match.group(1)) + 1
        if completed_epoch in CHECKPOINT_EPOCHS:
            destination = CHECKPOINT_DIR / f"{experiment_id}_{signature}_epoch{completed_epoch}.pt"
            shutil.copy2(periodic_path, destination)
            candidate_checkpoint_paths[completed_epoch] = destination

    # 마지막 100 epoch은 last.pt를 명시적으로 사용해 누락을 막는다.
    final_epoch = int(config["epochs"])
    if final_epoch in CHECKPOINT_EPOCHS:
        destination = CHECKPOINT_DIR / f"{experiment_id}_{signature}_epoch{final_epoch}.pt"
        shutil.copy2(last_source, destination)
        candidate_checkpoint_paths[final_epoch] = destination

    missing_candidate_epochs = sorted(set(CHECKPOINT_EPOCHS) - set(candidate_checkpoint_paths))
    if RUN_PROFILE == "standard" and missing_candidate_epochs:
        raise FileNotFoundError(
            f"Required competition checkpoints were not saved: {missing_candidate_epochs}. "
            f"Available: {sorted(path.name for path in weights_dir.glob('*.pt'))}"
        )

    # 필요한 20/40/60/100만 Drive checkpoint 폴더에 복사한 뒤, 로컬의 나머지 epoch checkpoint는 삭제해 용량을 회수한다.
    for periodic_path in weights_dir.glob("epoch*.pt"):
        periodic_path.unlink(missing_ok=True)

    convergence = analyze_yolo_convergence(results_path)
    save_checkpoint_manifest(checkpoint_path, manifest_path, manifest)

    trained_model = YOLO(str(checkpoint_path))
    validate_yolo_model_scale(trained_model, expected_scale, model_name)
    print(
        f"Completed {experiment_id}: best epoch {convergence['best_epoch']}, "
        f"validation mAP50-95 {convergence['best_val_map50_95']:.4f}."
    )
    return {
        "model": trained_model,
        "model_name": model_name,
        "checkpoint_path": checkpoint_path,
        "manifest_path": manifest_path,
        "manifest": manifest,
        "config": config,
        "checkpoint_reused": False,
        "convergence": convergence,
        "results_path": results_path,
        "candidate_checkpoint_paths": candidate_checkpoint_paths,
        "run_dir": run_dir,
    }

## 2-3. YOLO12m 단일 100 Epoch 학습 — 20/40/60/100 저장

In [ ]:
# YOLO12m 한 모델만 지정된 epoch까지 학습한다.
yolo12_experiments = {}
epoch_config = BASELINE_CONFIGS["YOLO12m"]
epoch_experiment_id = epoch_config["experiment_id"]
yolo12_result = run_timed_experiment(
    "YOLO12m",
    epoch_config,
    train_yolo_experiment,
    "YOLO12m",
    "yolo12m.pt",
    "m",
    epoch_config,
)
yolo12_experiments[epoch_experiment_id] = yolo12_result
print(f"YOLO12m epoch experiment completed: {epoch_experiment_id}")


Timer started: YOLO11s / yolo11s_competition_100e_single_trajectory / 2026-08-11T04:20:41.791982
YOLO11s scale check: s
Reused verified YOLO11s checkpoint: yolo11s_competition_100e_single_trajectory_bb534fcdb46be99f_best.pt
Timer completed: YOLO11s / yolo11s_competition_100e_single_trajectory / 00:00:01 / status=checkpoint_reused
Timer record: /content/drive/MyDrive/baby_kangaroo/파이프라인/yolo11s_competition_train_v4_0/timings/yolo11s_competition_100e_single_trajectory_bb534fcdb46be99f.json
YOLO11s epoch experiment completed: yolo11s_competition_100e_single_trajectory


# 3. Validation 평가


In [ ]:
def synchronize_if_cuda():
    """GPU 비동기 연산이 끝날 때까지 기다린다."""
    if torch.cuda.is_available():
        torch.cuda.synchronize()


def map_model_labels_to_original(labels, mapping, source_name):
    """YOLO 내부 라벨을 실제 original_category_id로 변환한다."""
    label_values = torch.as_tensor(labels).detach().cpu().reshape(-1).tolist()
    missing = sorted({int(value) for value in label_values} - set(mapping))
    if missing:
        raise KeyError(f"Unknown labels in {source_name}: {missing}")
    return torch.tensor([mapping[int(value)] for value in label_values], dtype=torch.int64)


def read_canonical_coco_ground_truth(file_name):
    """COCO 원본 ID·xyxy ground truth를 반환한다."""
    items = coco_original_annotations.get(file_name, [])
    boxes = [box_xyxy for _original_id, box_xyxy in items]
    labels = [original_id for original_id, _box_xyxy in items]
    return (
        torch.tensor(boxes, dtype=torch.float32).reshape(-1, 4),
        torch.tensor(labels, dtype=torch.int64).reshape(-1),
    )


INFERENCE_CACHE = {}
TEST_INFERENCE_CALLS = Counter()


def run_yolo_inference(result, split_name):
    """YOLO 결과에 공통 COCO ground truth를 붙여 공통 평가 형식으로 만든다."""
    model_name = result["model_name"]
    experiment_id = result["config"]["experiment_id"]
    cache_key = (
        model_name,
        experiment_id,
        split_name,
        str(result["checkpoint_path"]),
    )
    if cache_key in INFERENCE_CACHE:
        return INFERENCE_CACHE[cache_key]

    image_dir = LOCAL_YOLO_DATASET / "images" / split_name
    expected_files = split_files[split_name]
    predict_kwargs = {
        "source": str(image_dir),
        "conf": RAW_PREDICTION_CONFIDENCE,
        "iou": NMS_IOU_THRESHOLD,
        "max_det": MAX_DETECTIONS,
        "imgsz": IMAGE_SIZE,
        "device": 0 if torch.cuda.is_available() else "cpu",
        "verbose": False,
        "stream": True,
    }

    # 첫 장을 미리 추론해 CUDA 초기화 시간을 제외한다.
    warmup_kwargs = dict(predict_kwargs)
    warmup_kwargs.update({"source": str(image_dir / expected_files[0]), "stream": False})
    result["model"].predict(**warmup_kwargs)
    synchronize_if_cuda()

    started_at = time.perf_counter()
    raw_results = list(tqdm(
        result["model"].predict(**predict_kwargs),
        total=len(expected_files),
        desc=f"[3-1] {model_name} {split_name} inference",
        unit="image",
        bar_format=PROGRESS_BAR_FORMAT,
    ))
    synchronize_if_cuda()
    elapsed = time.perf_counter() - started_at

    result_by_file = {}
    for raw_result in raw_results:
        file_name = Path(raw_result.path).name
        if file_name in result_by_file:
            raise ValueError(f"Duplicate YOLO result file name: {file_name}")
        result_by_file[file_name] = raw_result
    if set(result_by_file) != set(expected_files):
        raise ValueError(f"{model_name} input and result file sets do not match.")

    ground_truth = []
    predictions = []
    for file_name in expected_files:
        gt_boxes, gt_labels = read_canonical_coco_ground_truth(file_name)
        raw_result = result_by_file[file_name]
        ground_truth.append({"image_id": file_name, "boxes": gt_boxes, "labels": gt_labels})
        predictions.append({
            "image_id": file_name,
            "boxes": raw_result.boxes.xyxy.detach().cpu().to(torch.float32).reshape(-1, 4),
            "labels": map_model_labels_to_original(
                raw_result.boxes.cls,
                YOLO_TO_ORIGINAL,
                f"{model_name} prediction",
            ),
            "scores": raw_result.boxes.conf.detach().cpu().to(torch.float32).reshape(-1),
        })

    metadata = {
        "latency_ms_per_image": elapsed * 1000 / max(len(ground_truth), 1),
        "head_mode": None,
    }
    inference_result = (ground_truth, predictions, metadata)
    INFERENCE_CACHE[cache_key] = inference_result
    if split_name == "test":
        TEST_INFERENCE_CALLS[model_name] += 1
    return inference_result

## 3-2. 공통 mAP·정밀도·재현율·F1·정확도

분류의 accuracy를 객체 탐지에 그대로 쓰면 배경 픽셀 수 때문에 의미가 흐려진다. 여기서는 정답 객체와 예측 객체를 IoU 0.50에서 1:1 매칭한 뒤 `TP / (TP + FP + FN)`을 **Detection Accuracy**로 정의한다. 같은 값은 객체 집합의 Jaccard 지수로도 볼 수 있다.

In [ ]:
# 모든 모델의 실제 원본 ID를 공통 평가용 0~55 ID로 바꾼다.
ORIGINAL_CLASS_INFO = {
    int(category["original_category_id"]): category["name"]
    for category in canonical_data["coco"]["categories"]
}
ORIGINAL_TO_EVAL = {
    original_id: index
    for index, original_id in enumerate(sorted(ORIGINAL_CLASS_INFO))
}


def align_records(ground_truth, predictions):
    """중복·누락을 검사하고 image_id로 ground truth와 prediction을 정렬한다."""
    if not ground_truth:
        raise ValueError("Ground truth is empty.")
    gt_by_id = {record["image_id"]: record for record in ground_truth}
    pred_by_id = {record["image_id"]: record for record in predictions}
    if len(gt_by_id) != len(ground_truth) or len(pred_by_id) != len(predictions):
        raise ValueError("Duplicate image IDs were found during evaluation.")
    if set(gt_by_id) != set(pred_by_id):
        raise ValueError("Ground truth and prediction image IDs do not match.")
    return [(gt_by_id[image_id], pred_by_id[image_id]) for image_id in sorted(gt_by_id)]


def remap_for_torchmetrics(labels):
    """실제 original_category_id를 TorchMetrics용 연속 0~55 값으로 바꾼다."""
    values = torch.as_tensor(labels).reshape(-1).tolist()
    missing = sorted({int(value) for value in values} - set(ORIGINAL_TO_EVAL))
    if missing:
        raise KeyError(f"Unknown original category IDs during evaluation: {missing}")
    return torch.tensor([ORIGINAL_TO_EVAL[int(value)] for value in values], dtype=torch.int64)


def compute_map_metrics(aligned_records):
    """표준 COCO 설정으로 mAP50-95, mAP50, mAP75, mAR100과 클래스별 AP를 계산한다."""
    ground_truth_count = sum(
        len(ground_truth["boxes"])
        for ground_truth, _ in aligned_records
    )
    prediction_count = sum(
        len(prediction["boxes"])
        for _, prediction in aligned_records
    )

    if ground_truth_count == 0:
        raise RuntimeError(
            "Validation ground truth is empty, so mAP cannot be calculated."
        )

    # 검증 이미지 전체에서 예측한 객체가 없으면 검출 성능을 0으로 기록한다.
    if prediction_count == 0:
        observed_classes = sorted({
            int(label)
            for ground_truth, _ in aligned_records
            for label in remap_for_torchmetrics(ground_truth["labels"]).tolist()
        })
        print(
            "No validation objects were detected. "
            "Validation mAP is recorded as 0.0."
        )
        return {
            "map": torch.tensor(0.0, dtype=torch.float32),
            "map_50": torch.tensor(0.0, dtype=torch.float32),
            "map_75": torch.tensor(0.0, dtype=torch.float32),
            f"mar_{COCO_EVAL_MAX_DETECTIONS}": torch.tensor(0.0, dtype=torch.float32),
            "classes": torch.tensor(observed_classes, dtype=torch.int32),
            "map_per_class": torch.zeros(len(observed_classes), dtype=torch.float32),
        }

    metric = MeanAveragePrecision(
        box_format="xyxy",
        iou_type="bbox",
        max_detection_thresholds=[1, 10, COCO_EVAL_MAX_DETECTIONS],
        class_metrics=True,
        average="macro",
    )
    predictions_tm = []
    targets_tm = []
    for ground_truth, prediction in aligned_records:
        targets_tm.append({
            "boxes": torch.as_tensor(
                ground_truth["boxes"], dtype=torch.float32
            ).reshape(-1, 4),
            "labels": remap_for_torchmetrics(ground_truth["labels"]),
        })
        predictions_tm.append({
            "boxes": torch.as_tensor(
                prediction["boxes"], dtype=torch.float32
            ).reshape(-1, 4),
            "labels": remap_for_torchmetrics(prediction["labels"]),
            "scores": torch.as_tensor(
                prediction["scores"], dtype=torch.float32
            ).reshape(-1),
        })

    metric.update(predictions_tm, targets_tm)
    result = metric.compute()
    map_value = float(result["map"].cpu().item())
    if not math.isfinite(map_value) or map_value < 0:
        raise RuntimeError(
            "Invalid mAP result with non-empty predictions. "
            f"mAP={map_value}, "
            f"ground_truth_boxes={ground_truth_count}, "
            f"prediction_boxes={prediction_count}"
        )
    return result


def fixed_threshold_counts(aligned_records, confidence_threshold):
    """confidence·class·IoU 조건으로 GT와 prediction을 1:1 매칭해 TP·FP·FN을 센다."""
    counts_by_original = {
        original_id: {"tp": 0, "fp": 0, "fn": 0, "support": 0}
        for original_id in ORIGINAL_CLASS_INFO
    }

    for ground_truth, prediction in aligned_records:
        gt_boxes = torch.as_tensor(ground_truth["boxes"], dtype=torch.float32).reshape(-1, 4)
        gt_labels = torch.as_tensor(ground_truth["labels"], dtype=torch.int64).reshape(-1)
        pred_boxes = torch.as_tensor(prediction["boxes"], dtype=torch.float32).reshape(-1, 4)
        pred_labels = torch.as_tensor(prediction["labels"], dtype=torch.int64).reshape(-1)
        pred_scores = torch.as_tensor(prediction["scores"], dtype=torch.float32).reshape(-1)

        keep = pred_scores >= float(confidence_threshold)
        pred_boxes = pred_boxes[keep]
        pred_labels = pred_labels[keep]
        pred_scores = pred_scores[keep]

        for original_id in ORIGINAL_CLASS_INFO:
            gt_indices = torch.where(gt_labels == original_id)[0]
            pred_indices = torch.where(pred_labels == original_id)[0]
            counts_by_original[original_id]["support"] += len(gt_indices)
            if len(pred_indices):
                pred_indices = pred_indices[torch.argsort(pred_scores[pred_indices], descending=True)]

            unmatched_gt = set(range(len(gt_indices)))
            true_positive = 0
            false_positive = 0
            for pred_index in pred_indices.tolist():
                if not unmatched_gt:
                    false_positive += 1
                    continue
                candidate_positions = sorted(unmatched_gt)
                candidate_boxes = gt_boxes[gt_indices[candidate_positions]]
                ious = box_iou(pred_boxes[pred_index].reshape(1, 4), candidate_boxes).reshape(-1)
                best_position = int(torch.argmax(ious).item())
                if float(ious[best_position].item()) >= EVAL_IOU_THRESHOLD:
                    unmatched_gt.remove(candidate_positions[best_position])
                    true_positive += 1
                else:
                    false_positive += 1

            counts_by_original[original_id]["tp"] += true_positive
            counts_by_original[original_id]["fp"] += false_positive
            counts_by_original[original_id]["fn"] += len(unmatched_gt)
    return counts_by_original


def summarize_fixed_counts(counts_by_original):
    """클래스별 TP·FP·FN을 micro, macro와 Detection Accuracy로 요약한다."""
    total_tp = sum(values["tp"] for values in counts_by_original.values())
    total_fp = sum(values["fp"] for values in counts_by_original.values())
    total_fn = sum(values["fn"] for values in counts_by_original.values())

    micro_precision = total_tp / (total_tp + total_fp) if total_tp + total_fp else 0.0
    micro_recall = total_tp / (total_tp + total_fn) if total_tp + total_fn else 0.0
    micro_f1 = (
        2 * micro_precision * micro_recall / (micro_precision + micro_recall)
        if micro_precision + micro_recall else 0.0
    )
    detection_accuracy = (
        total_tp / (total_tp + total_fp + total_fn)
        if total_tp + total_fp + total_fn else 0.0
    )

    class_rows = []
    for original_id, class_name in ORIGINAL_CLASS_INFO.items():
        values = counts_by_original[original_id]
        tp, fp, fn = values["tp"], values["fp"], values["fn"]
        precision = tp / (tp + fp) if tp + fp else np.nan
        recall = tp / (tp + fn) if tp + fn else np.nan
        f1 = (
            2 * precision * recall / (precision + recall)
            if np.isfinite(precision) and np.isfinite(recall) and precision + recall else np.nan
        )
        class_rows.append({
            "original_category_id": original_id,
            "class_name": class_name,
            "support": values["support"],
            "tp": tp,
            "fp": fp,
            "fn": fn,
            "precision": precision,
            "recall": recall,
            "f1": f1,
            "status": "observed" if values["support"] > 0 else "not_observed",
        })

    class_table = pd.DataFrame(class_rows)
    observed = class_table[class_table["support"] > 0]
    summary = {
        "micro_precision": micro_precision,
        "micro_recall": micro_recall,
        "micro_f1": micro_f1,
        "macro_precision_observed": float(observed["precision"].fillna(0).mean()),
        "macro_recall_observed": float(observed["recall"].fillna(0).mean()),
        "macro_f1_observed": float(observed["f1"].fillna(0).mean()),
        "detection_accuracy": detection_accuracy,
        "tp": int(total_tp),
        "fp": int(total_fp),
        "fn": int(total_fn),
        "observed_classes": int(len(observed)),
        "total_classes": EXPECTED_NUM_CLASSES,
        "class_coverage": len(observed) / EXPECTED_NUM_CLASSES,
    }
    return summary, class_table


def select_confidence_on_validation(ground_truth, predictions):
    """Validation micro F1이 가장 높은 confidence를 선택해 Test threshold 누수를 막는다."""
    aligned = align_records(ground_truth, predictions)
    rows = []
    for threshold in np.arange(0.05, 0.951, 0.05):
        counts = fixed_threshold_counts(aligned, threshold)
        summary, _class_table = summarize_fixed_counts(counts)
        rows.append({
            "threshold": float(threshold),
            "micro_precision": summary["micro_precision"],
            "micro_recall": summary["micro_recall"],
            "micro_f1": summary["micro_f1"],
            "detection_accuracy": summary["detection_accuracy"],
        })
    threshold_table = pd.DataFrame(rows)
    best_row = threshold_table.sort_values(
        ["micro_f1", "micro_precision", "threshold"],
        ascending=[False, False, True],
    ).iloc[0]
    return float(best_row["threshold"]), threshold_table


def evaluate_records(ground_truth, predictions, threshold, latency_ms):
    """한 분할에서 threshold 독립 mAP와 선택 threshold의 고정 운영점 지표를 계산한다."""
    aligned = align_records(ground_truth, predictions)
    map_result = compute_map_metrics(aligned)
    counts = fixed_threshold_counts(aligned, threshold)
    fixed_summary, class_table = summarize_fixed_counts(counts)

    common_counts = fixed_threshold_counts(aligned, COMMON_CONFIDENCE_THRESHOLD)
    common_summary, _common_class_table = summarize_fixed_counts(common_counts)
    summary = {
        "mAP50_95": float(map_result["map"].cpu().item()),
        "mAP50": float(map_result["map_50"].cpu().item()),
        "mAP75": float(map_result["map_75"].cpu().item()),
        f"mAR{COCO_EVAL_MAX_DETECTIONS}": float(
            map_result[f"mar_{COCO_EVAL_MAX_DETECTIONS}"].cpu().item()
        ),
        "selected_confidence": float(threshold),
        **fixed_summary,
        "f1_at_common_0_25": common_summary["micro_f1"],
        "precision_at_common_0_25": common_summary["micro_precision"],
        "recall_at_common_0_25": common_summary["micro_recall"],
        "latency_ms_per_image": float(latency_ms),
    }

    per_class_ap = {
        sorted(ORIGINAL_CLASS_INFO)[int(eval_id)]: float(ap)
        for eval_id, ap in zip(
            map_result["classes"].cpu().tolist(),
            map_result["map_per_class"].cpu().tolist(),
        )
    }
    class_table["AP50_95"] = class_table["original_category_id"].map(per_class_ap)
    return summary, class_table, aligned


def bootstrap_fixed_metric_intervals(aligned_records, threshold, iterations=BOOTSTRAP_ITERATIONS):
    """Test 이미지 단위 bootstrap으로 주요 운영점 지표의 95% 구간을 계산한다."""
    rng = np.random.default_rng(SEED)
    metric_values = defaultdict(list)
    record_count = len(aligned_records)
    for _iteration in tqdm(
        range(iterations),
        desc="[3-2] Bootstrap confidence intervals",
        unit="sample",
        leave=False,
        bar_format=PROGRESS_BAR_FORMAT,
    ):
        sampled_indices = rng.integers(0, record_count, size=record_count)
        sampled_records = [aligned_records[int(index)] for index in sampled_indices]
        counts = fixed_threshold_counts(sampled_records, threshold)
        summary, _class_table = summarize_fixed_counts(counts)
        for metric_name in ["micro_precision", "micro_recall", "micro_f1", "detection_accuracy"]:
            metric_values[metric_name].append(summary[metric_name])

    intervals = {}
    for metric_name, values in metric_values.items():
        intervals[f"{metric_name}_ci_low"] = float(np.quantile(values, 0.025))
        intervals[f"{metric_name}_ci_high"] = float(np.quantile(values, 0.975))
    return intervals


def count_model_parameters(result):
    """모델의 전체 parameter 수를 같은 방식으로 계산한다."""
    model = result["model"].model
    return int(sum(parameter.numel() for parameter in model.parameters()))

# ================= V4 COMPETITION METRIC / POLICY =================
def _competition_metric_from_aligned(aligned_records):
    metric = MeanAveragePrecision(
        box_format="xyxy",
        iou_type="bbox",
        iou_thresholds=COMPETITION_IOU_THRESHOLDS,
        max_detection_thresholds=[1, 10, COCO_EVAL_MAX_DETECTIONS],
        class_metrics=True,
    )
    predictions_tm, targets_tm = [], []
    for ground_truth, prediction in aligned_records:
        targets_tm.append({
            "boxes": torch.as_tensor(ground_truth["boxes"], dtype=torch.float32).reshape(-1, 4),
            "labels": remap_for_torchmetrics(ground_truth["labels"]),
        })
        predictions_tm.append({
            "boxes": torch.as_tensor(prediction["boxes"], dtype=torch.float32).reshape(-1, 4),
            "labels": remap_for_torchmetrics(prediction["labels"]),
            "scores": torch.as_tensor(prediction["scores"], dtype=torch.float32).reshape(-1),
        })
    metric.update(predictions_tm, targets_tm)
    return metric.compute()


def apply_prediction_policy(predictions, confidence_threshold, top_k):
    filtered = []
    for record in predictions:
        boxes = torch.as_tensor(record["boxes"], dtype=torch.float32).reshape(-1, 4)
        labels = torch.as_tensor(record["labels"], dtype=torch.int64).reshape(-1)
        scores = torch.as_tensor(record["scores"], dtype=torch.float32).reshape(-1)
        keep = torch.where(scores >= float(confidence_threshold))[0]
        if len(keep):
            keep = keep[torch.argsort(scores[keep], descending=True)]
            if top_k is not None:
                keep = keep[:int(top_k)]
        filtered.append({
            "image_id": record["image_id"],
            "boxes": boxes[keep], "labels": labels[keep], "scores": scores[keep],
        })
    return filtered


def competition_map_for_policy(ground_truth, predictions, threshold, top_k):
    filtered = apply_prediction_policy(predictions, threshold, top_k)
    aligned = align_records(ground_truth, filtered)
    result = _competition_metric_from_aligned(aligned)
    return float(result["map"].cpu().item())


def select_competition_policy(ground_truth, predictions):
    rows = []
    for threshold in CONFIDENCE_CANDIDATES:
        for top_k in TOP_K_CANDIDATES:
            rows.append({
                "threshold": float(threshold),
                "top_k": top_k,
                "top_k_label": "unlimited" if top_k is None else str(int(top_k)),
                "competition_mAP": competition_map_for_policy(
                    ground_truth, predictions, threshold, top_k
                ),
            })
    table = pd.DataFrame(rows)
    table["top_k_sort"] = table["top_k"].apply(lambda value: 10**9 if pd.isna(value) else int(value))
    best = table.sort_values(
        ["competition_mAP", "threshold", "top_k_sort"],
        ascending=[False, True, True],
    ).iloc[0]
    selected_top_k = None if pd.isna(best["top_k"]) else int(best["top_k"])
    return float(best["threshold"]), selected_top_k, table.drop(columns="top_k_sort")


def evaluate_checkpoint_competition(result, checkpoint_path):
    candidate = dict(result)
    candidate["model"] = YOLO(str(checkpoint_path))
    candidate["checkpoint_path"] = Path(checkpoint_path)
    ground_truth, predictions, metadata = run_yolo_inference(candidate, "val")
    threshold, top_k, policy_table = select_competition_policy(ground_truth, predictions)
    score = competition_map_for_policy(ground_truth, predictions, threshold, top_k)
    # 기존 evaluate_records는 진단 지표 보존용이다.
    diagnostic_summary, class_table, aligned = evaluate_records(
        ground_truth, predictions, threshold, metadata["latency_ms_per_image"]
    )
    return {
        "competition_mAP": score,
        "selected_confidence": threshold,
        "selected_top_k": top_k,
        "policy_table": policy_table,
        "diagnostic_summary": diagnostic_summary,
        "class_table": class_table,
        "ground_truth": ground_truth,
        "predictions": predictions,
        "aligned": aligned,
        "latency_ms_per_image": metadata["latency_ms_per_image"],
    }


In [ ]:
# 20/40/60/100 checkpoint를 같은 Validation과 같은 competition evaluator로 비교한다.
if RUN_PROFILE == "smoke":
    checkpoint_paths = {1: yolo12_result["checkpoint_path"]}
else:
    checkpoint_paths = dict(yolo12_result.get("candidate_checkpoint_paths", {}))
    # 재실행으로 last checkpoint를 재사용한 경우에도 20/40/60/100 파일을 다시 발견한다.
    if not checkpoint_paths:
        experiment_id = yolo12_result["config"]["experiment_id"]
        signature = yolo12_result["manifest"]["run_signature"]
        for completed_epoch in CHECKPOINT_EPOCHS:
            candidate = CHECKPOINT_DIR / f"{experiment_id}_{signature}_epoch{completed_epoch}.pt"
            if candidate.is_file():
                checkpoint_paths[int(completed_epoch)] = candidate
    missing = sorted(set(CHECKPOINT_EPOCHS) - set(checkpoint_paths))
    if missing:
        raise FileNotFoundError(f"Missing required checkpoint epochs: {missing}")

checkpoint_evaluations = {}
checkpoint_rows = []
for completed_epoch, checkpoint_path in sorted(checkpoint_paths.items()):
    detail = evaluate_checkpoint_competition(yolo12_result, checkpoint_path)
    checkpoint_evaluations[int(completed_epoch)] = detail
    diagnostic = detail["diagnostic_summary"]
    checkpoint_rows.append({
        "completed_epoch": int(completed_epoch),
        "checkpoint_path": str(checkpoint_path),
        "checkpoint_sha256": sha256_file(checkpoint_path),
        "competition_mAP": detail["competition_mAP"],
        "selected_confidence": detail["selected_confidence"],
        "selected_top_k": detail["selected_top_k"],
        "diagnostic_mAP50_95": diagnostic["mAP50_95"],
        "mAP75": diagnostic["mAP75"],
        "micro_precision": diagnostic["micro_precision"],
        "micro_recall": diagnostic["micro_recall"],
        "micro_f1": diagnostic["micro_f1"],
        "latency_ms_per_image": detail["latency_ms_per_image"],
    })

checkpoint_comparison_table = pd.DataFrame(checkpoint_rows).sort_values(
    ["competition_mAP", "completed_epoch"], ascending=[False, True]
).reset_index(drop=True)
display(checkpoint_comparison_table)

best_row = checkpoint_comparison_table.iloc[0]
SELECTED_EPOCH = int(best_row["completed_epoch"])
SELECTED_CHECKPOINT_PATH = Path(best_row["checkpoint_path"])
SELECTED_CONFIDENCE = float(best_row["selected_confidence"])
SELECTED_TOP_K = None if pd.isna(best_row["selected_top_k"]) else int(best_row["selected_top_k"])
selected_detail = checkpoint_evaluations[SELECTED_EPOCH]

print(
    f"Selected YOLO12m checkpoint: epoch={SELECTED_EPOCH}, "
    f"competition mAP={best_row['competition_mAP']:.5f}, "
    f"confidence={SELECTED_CONFIDENCE}, top_k={SELECTED_TOP_K}"
)


[3-1] YOLO11s val inference:   0%|          | 0/38 [00:00<?, ?image/s]

[3-1] YOLO11s val inference:   0%|          | 0/38 [00:00<?, ?image/s]

[3-1] YOLO11s val inference:   0%|          | 0/38 [00:00<?, ?image/s]

[3-1] YOLO11s val inference:   0%|          | 0/38 [00:00<?, ?image/s]

,completed_epoch,checkpoint_path,checkpoint_sha256,competition_mAP,selected_confidence,selected_top_k,diagnostic_mAP50_95,mAP75,micro_precision,micro_recall,micro_f1,latency_ms_per_image
0,60,/content/drive/MyDrive/baby_kangaroo/파이프라인/yol...,95264ab24ff9ef5f396ed70adbc4c60c15d25cfb4f340e...,0.905796,0.001,4.0,0.915896,0.925996,0.561905,0.95935,0.708709,63.934188
1,100,/content/drive/MyDrive/baby_kangaroo/파이프라인/yol...,1c3dcdf9e0cc26c865f4492f0603b60b0d0134cb511bfd...,0.899207,0.001,4.0,0.912601,0.925996,0.605128,0.95935,0.742138,74.797192
2,40,/content/drive/MyDrive/baby_kangaroo/파이프라인/yol...,aab0ad67ed65e5bd67f9cf5c851d54ee6ff18b428bf02d...,0.897761,0.001,6.0,0.923684,0.949606,0.437956,0.97561,0.604534,57.494709
3,20,/content/drive/MyDrive/baby_kangaroo/파이프라인/yol...,1771b87b92a0e27546d3d162f7564c5019d307eddedf9e...,0.872674,0.001,NaN,0.917208,0.961742,0.242485,0.98374,0.389068,127.679987


Selected YOLO11s checkpoint: epoch=60, competition mAP=0.90580, confidence=0.001, top_k=4


# 4. 결과 저장


In [ ]:
# V4 결과 저장: 20/40/60/100 비교표와 최종 제출 정책을 보존한다.
report_id = f"{split_fingerprint[:16]}_{yolo12_result['manifest']['run_signature']}"
comparison_csv_path = REPORT_DIR / f"yolo12m_checkpoint_20_40_60_100_{report_id}.csv"
policy_csv_path = REPORT_DIR / f"yolo12m_selected_policy_grid_{report_id}.csv"
report_path = REPORT_DIR / f"yolo12m_competition_report_{report_id}.json"
policy_path = REPORT_DIR / "yolo12m_competition_selected_policy.json"

checkpoint_comparison_table.to_csv(comparison_csv_path, index=False, encoding="utf-8-sig")
selected_detail["policy_table"].to_csv(policy_csv_path, index=False, encoding="utf-8-sig")

# 최종 전체 labeled data 학습은 선택 epoch와 정책이 확정된 뒤 실행한다. (현재 스냅샷 기준 232장)
FINAL_FULL_DATA_CHECKPOINT = None
if RUN_FULL_DATA_FINAL_TRAIN:
    all_file_names = sorted(
        set(split_files["train"]) | set(split_files["val"]) | set(split_files["test"])
    )
    if len(all_file_names) != EXPECTED_IMAGES:
        raise RuntimeError(f"Full-data file count mismatch: {len(all_file_names)}")

    # 기존 split 폴더를 억지로 합치지 않고 전체 labeled data 전용 로컬 YOLO dataset을 새로 만든다.
    full_dataset_root = LOCAL_WORK_ROOT / f"yolo_full_{len(all_file_names)}"
    reset_owned_directory(full_dataset_root, LOCAL_WORK_ROOT)
    full_image_dir = full_dataset_root / "images" / "train"
    full_label_dir = full_dataset_root / "labels" / "train"
    full_image_dir.mkdir(parents=True, exist_ok=True)
    full_label_dir.mkdir(parents=True, exist_ok=True)

    for file_name in tqdm(all_file_names, desc=f"Build full {len(all_file_names)} YOLO dataset", unit="image"):
        source_image = LOCAL_IMAGE_ROOT / file_name
        destination_image = full_image_dir / file_name
        try:
            os.link(source_image, destination_image)
        except OSError:
            shutil.copy2(source_image, destination_image)
        write_canonical_yolo_label(
            file_name,
            full_label_dir / f"{Path(file_name).stem}.txt",
            ORIGINAL_TO_YOLO,
        )

    full_yaml = full_dataset_root / f"data_full_{len(all_file_names)}.yaml"
    with full_yaml.open("w", encoding="utf-8") as file:
        yaml.safe_dump({
            "path": str(full_dataset_root),
            "train": "images/train",
            # Ultralytics dataset schema를 위해 val key는 유지하지만 train 호출에서 val=False로 비활성화한다.
            "val": "images/train",
            "names": YOLO_NAMES,
            "nc": EXPECTED_NUM_CLASSES,
        }, file, allow_unicode=True, sort_keys=False)

    full_run_name = f"yolo12m_full{len(all_file_names)}_100e_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
    full_model = YOLO("yolo12m.pt")
    augmentation_arguments = yolo_augmentation_arguments(yolo12_result["config"]["augmentation"])
    full_model.train(
        data=str(full_yaml),
        epochs=100,
        batch=yolo12_result["config"]["batch_size"],
        imgsz=IMAGE_SIZE,
        device=0 if torch.cuda.is_available() else "cpu",
        optimizer="SGD",
        lr0=yolo12_result["config"]["learning_rate"],
        lrf=0.10,
        momentum=yolo12_result["config"]["momentum"],
        weight_decay=yolo12_result["config"]["weight_decay"],
        warmup_epochs=2.0 if yolo12_result["config"]["stage"] == "tuning" else 1.0,
        cos_lr=True,
        patience=100,
        max_det=MAX_DETECTIONS,
        seed=SEED,
        deterministic=True,
        workers=2,
        cache=False,
        amp=USE_AMP,
        val=False,
        project=str(LOCAL_RUNS_ROOT),
        name=full_run_name,
        exist_ok=False,
        plots=False,
        save=True,
        save_period=1,
        verbose=True,
        **augmentation_arguments,
    )
    full_run_dir = Path(full_model.trainer.save_dir)
    weights_dir = full_run_dir / "weights"
    if SELECTED_EPOCH == 100:
        source = weights_dir / "last.pt"
    else:
        source = weights_dir / f"epoch{SELECTED_EPOCH - 1}.pt"
    if not source.is_file():
        raise FileNotFoundError(f"Selected full-data epoch checkpoint not found: {source}")
    FINAL_FULL_DATA_CHECKPOINT = CHECKPOINT_DIR / f"yolo12m_full{len(all_file_names)}_selected_epoch{SELECTED_EPOCH}.pt"
    shutil.copy2(source, FINAL_FULL_DATA_CHECKPOINT)
    # 필요한 full-data checkpoint만 보존하고 100개의 로컬 periodic 파일은 정리한다.
    for periodic_path in weights_dir.glob("epoch*.pt"):
        periodic_path.unlink(missing_ok=True)
    print(f"Full-data selected checkpoint: {FINAL_FULL_DATA_CHECKPOINT}")

selection_contract = {
    "schema_version": 4,
    "pipeline_version": PIPELINE_VERSION,
    "competition_metric": COMPETITION_METRIC,
    "competition_iou_thresholds": COMPETITION_IOU_THRESHOLDS,
    "created_at": datetime.now().isoformat(),
    "dataset_fingerprint": canonical_data["dataset_fingerprint"],
    "split_fingerprint": split_fingerprint,
    "training_trajectory_epochs": 100,
    "compared_epochs": list(CHECKPOINT_EPOCHS),
    "selected_epoch": SELECTED_EPOCH,
    "selected_checkpoint_path": str(SELECTED_CHECKPOINT_PATH),
    "selected_checkpoint_sha256": sha256_file(SELECTED_CHECKPOINT_PATH),
    "selected_confidence": SELECTED_CONFIDENCE,
    "selected_top_k": SELECTED_TOP_K,
    "selected_competition_mAP": float(best_row["competition_mAP"]),
    "diagnostic_mAP50_95": float(best_row["diagnostic_mAP50_95"]),
    "full_data_training_enabled": bool(RUN_FULL_DATA_FINAL_TRAIN),
    "full_data_checkpoint_path": str(FINAL_FULL_DATA_CHECKPOINT) if FINAL_FULL_DATA_CHECKPOINT else None,
    "full_data_checkpoint_sha256": sha256_file(FINAL_FULL_DATA_CHECKPOINT) if FINAL_FULL_DATA_CHECKPOINT else None,
    "comparison_csv": str(comparison_csv_path),
    "policy_grid_csv": str(policy_csv_path),
}
write_json_atomic(report_path, selection_contract)
write_json_atomic(policy_path, selection_contract)

quality_checks = pd.DataFrame([
    {"check": "Single 100-epoch trajectory", "status": "PASS", "evidence": 100},
    {"check": "Required checkpoints compared", "status": "PASS" if RUN_PROFILE == "smoke" or set(CHECKPOINT_EPOCHS) <= set(checkpoint_evaluations) else "FAIL", "evidence": sorted(checkpoint_evaluations)},
    {"check": "Competition metric used", "status": "PASS", "evidence": COMPETITION_METRIC},
    {"check": "Confidence selected by competition mAP", "status": "PASS", "evidence": SELECTED_CONFIDENCE},
    {"check": "Top-K selected by competition mAP", "status": "PASS", "evidence": SELECTED_TOP_K},
    {"check": "Selected checkpoint exists", "status": "PASS" if SELECTED_CHECKPOINT_PATH.is_file() else "FAIL", "evidence": str(SELECTED_CHECKPOINT_PATH)},
])
display(quality_checks)
if (quality_checks["status"] != "PASS").any():
    raise RuntimeError("YOLO12m Competition Train V4 quality checks failed.")

print(f"Checkpoint comparison: {comparison_csv_path}")
print(f"Selection policy: {policy_path}")
print(f"Report: {report_path}")
if not RUN_FULL_DATA_FINAL_TRAIN:
    print("After hyperparameters are final, set RUN_FULL_DATA_FINAL_TRAIN=True and rerun the final-training cell.")


Build full 232 YOLO dataset:   0%|          | 0/232 [00:00<?, ?image/s]

New https://pypi.org/project/ultralytics/8.4.117 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.116 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=2, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/content/baby_kangaroo_yolo11s_competition_v4_0/yolo_full_232/data_full_232.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.0, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.0, hsv_s=0.0, hsv_v=0.0, imgsz=960, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.005, lrf=0.1,

,check,status,evidence
0,Single 100-epoch trajectory,PASS,100
1,Required checkpoints compared,PASS,"[20, 40, 60, 100]"
2,Competition metric used,PASS,mAP@[0.75:0.95]
3,Confidence selected by competition mAP,PASS,0.001
4,Top-K selected by competition mAP,PASS,4
5,Selected checkpoint exists,PASS,/content/drive/MyDrive/baby_kangaroo/파이프라인/yol...


Checkpoint comparison: /content/drive/MyDrive/baby_kangaroo/파이프라인/yolo11s_competition_train_v4_0/reports/yolo11s_checkpoint_20_40_60_100_3b914e2658165daa_bb534fcdb46be99f.csv
Selection policy: /content/drive/MyDrive/baby_kangaroo/파이프라인/yolo11s_competition_train_v4_0/reports/yolo11s_competition_selected_policy.json
Report: /content/drive/MyDrive/baby_kangaroo/파이프라인/yolo11s_competition_train_v4_0/reports/yolo11s_competition_report_3b914e2658165daa_bb534fcdb46be99f.json
